# Notebook 09: Deliverables — Manuscript Outputs

**Pipeline layer J.** Renders the figures, tables, headline numbers, and
caption seeds for the law-review manuscript on the empirical computational
framework for scènes à faire. Organized by manuscript section (Part II
through Appendix B). No new scientific computation; no API calls; no GPU.
Total runtime under five minutes on Colab CPU.

## Purpose

This notebook is the rendering layer that turns the project's parquet
outputs into the artifacts the Overleaf manuscript needs. Every output
here traces to a specific upstream parquet through the `manifest.csv`
that the notebook writes. When an upstream notebook is rerun and its
parquet regenerated, rerunning this notebook propagates the change
into the manuscript automatically.

Outputs are named by manuscript section. A figure named `F-IV-1` goes
in Part IV of the manuscript, position 1. A table named `T-VII-2` goes
in Part VII, position 2. The numbering matches the section the paper
will use, so a writer can drop `\input{T-IV-1_top10_per_genre.tex}`
into the right place without rebinding.

## Manuscript section → outputs map

```
Part I.   Introduction                  — narrative prose, no figures
Part II.  Doctrinal background          — T-II-1 (descriptor mapping)
Part III. Methodology                   — T-III-1 (corpus)
                                          T-III-2 (extraction summary)
                                          T-III-3 (LLM validation metrics)
                                          F-III-1 (pipeline flow diagram)
Part IV.  Core results                  — T-IV-1 (top-10 per genre, HEADLINE)
                                          T-IV-2 (10-category taxonomy)
                                          F-IV-1 (measure-correlation heatmap)
                                          F-IV-2 (within-genre ensemble distributions)
                                          F-IV-3 (pre/post-filter top-K composition)
Part V.   Validation                    — T-V-1 (per-genre partial r, surprisal)
                                          T-V-2 (per-genre Cohen's d, Arm 1)
                                          T-V-3 (per-genre Cohen's d, Arm 2)
                                          F-V-1 (length-confound 2-panel)
                                          F-V-2 (Arm 1 Cohen's d per genre)
                                          F-V-3 (Arm 2 Cohen's d per genre)
Part VI.  Combinations and networks     — T-VI-1 (top-5 combinations per genre)
                                          F-VI-1 (Western co-occurrence network)
                                          F-VI-2 (centrality vs stockiness scatter)
Part VII. Temporal dynamics             — T-VII-1 (vocabulary change-points)
                                          T-VII-2 (emerging conventions by bin)
                                          F-VII-1 (per-genre bin means over time)
                                          F-VII-2 (canonical-element lifecycles)
                                          F-VII-3 (per-genre survival curves)
Part VIII.Litigation tool               — T-VIII-1 (candidate scènes à faire per genre)
                                          T-VIII-2 (worked example: Western case)
Part IX.  Discussion                    — T-IX-1 (Animation anomaly diagnostic)
Appendix A. Supplementary tables        — T-App-A1..A4 (top-25 per element type)
Appendix B. Supplementary figures       — F-App-B1..B17 (per-genre networks)
                                          F-App-B18..B34 (per-genre lifecycles)
```

## Inputs

- `v4_rebuild/01c_union/movies_union.parquet` — 45,958 films
- `v4_rebuild/02_elements/element_index.parquet` — 5,658,714 instances (post-canonicalization)
- `v4_rebuild/02b_llm_validation/agreement_metrics.parquet` — 1,254 validated films
- `v4_rebuild/03_stockiness/stockiness_long.parquet` — 345,366 cells, 27,212 unique pairs (after the October 2026 filter correction)
- `v4_rebuild/03_stockiness/top_stock_by_genre_{overall,nouns,chunks,verbs,entities}.parquet`
- `v4_rebuild/03b_classification/element_classifications.parquet` — 40,877 classifications made on the May 2026 vocabulary (the 27,212 pairs in the current table are a subset), 94.5% gold-set accuracy
- `v4_rebuild/03b_classification/stockiness_long_classified.parquet` — canonical enriched table
- `v4_rebuild/03c_network/element_centrality_by_genre.parquet` — 17 per-genre centralities
- `v4_rebuild/03c_network/combination_stockiness.parquet` — 9.2M element combinations
- `v4_rebuild/04_surprisal/per_film_surprisal.parquet` + `correlation_table.parquet`
- `v4_rebuild/04_surprisal/per_film_mean_stockiness.parquet`
- `v4_rebuild/05_embeddings/element_embeddings.{npy,index.parquet}` — voyage-4-large
- `v4_rebuild/05_embeddings/semantic_validation.parquet` (Arm 1)
- `v4_rebuild/05_embeddings/filter_validation.parquet` (Arm 2)
- `v4_rebuild/05_embeddings/genre_centroids.parquet`
- `v4_rebuild/08_temporal/stockiness_by_bin.parquet` — 815,444 per-bin cells
- `v4_rebuild/08_temporal/element_lifecycle.parquet`
- `v4_rebuild/08_temporal/change_points.parquet`
- `v4_rebuild/08_temporal/survival_curves.parquet`

## Outputs

Under `v4_rebuild/09_deliverables/`:

- `figures/pdf/` — vector PDFs, filenames carry section identifiers
- `figures/png/` — 300-DPI raster previews
- `figures/source_data/` — CSV behind each plot
- `tables/latex/` — `\input`-ready booktabs tables
- `tables/csv/` — CSV mirror for archival
- `headline_numbers.json` and `headline_numbers.tex` — macro file
- `caption_seeds.tex` — ready-to-paste `\caption{...}` strings per figure / table
- `paper_section_index.md` — section → outputs reference for the manuscript
- `manifest.csv` — provenance trail
- `build_summary.txt` — inventory and full headline listing

## Cost and runtime

No API spend. No GPU. Pure pandas, matplotlib, networkx. Expected runtime
five to ten minutes on Colab CPU. The per-genre fan-outs (17 networks +
17 lifecycle panels) are the slowest steps.

## Key design choices

- **Section-aware filenames.** `T-IV-1_top10_per_genre.tex` declares its
  manuscript location. The Overleaf manuscript references each output by
  its identifier; no manual reshuffling needed when sections are reordered.
- **Caption seeds in a separate file.** `caption_seeds.tex` contains
  ready-to-paste `\caption` strings so the manuscript writer can drop
  consistent prose into each figure environment without rewriting from
  scratch.
- **Franchise blocklist updated for post-canonicalization surface forms.**
  Canonicalization renamed some franchise terms (`stooges` → `stooge`,
  `germans` → `german`). The blocklist tracks the current surface forms.
  Entries are documented with one-line rationales in the setup cell so
  reviewers can audit the choices.
- **Multi-POS dedup applied at display only.** The underlying pipeline
  retains multi-POS variants as separate cells (per the Bug 4 design
  decision in CLAUDE.md). For body tables in the manuscript, the highest-
  ensemble row per (genre, surface form) is shown to avoid display
  redundancy.


In [14]:
# ----------------------------------------------------------------------
# Setup and configuration
#
# Colab Drive mount, imports, paths, plotting defaults, the fixed
# 17-color genre palette, the franchise blocklist, the lifecycle pick
# list for F-VII-2, and the worked-example case definition.
# ----------------------------------------------------------------------

!pip install -q umap-learn

import os, json, re, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patches as mpatch
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import networkx as nx
from tqdm.auto import tqdm

# --- Google Drive (idempotent mount) ----------------------------------
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('  Google Drive already mounted; skipping remount.')

# --- Paths ------------------------------------------------------------
ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
            'DGK Lab - Scene A Faire Computational')
V4 = ROOT / 'Analysis/v4_rebuild'

# Upstream parquets
PATH_MOVIES        = V4 / '01c_union'           / 'movies_union.parquet'
PATH_ELEM_IDX      = V4 / '02_elements'         / 'element_index.parquet'
PATH_AGREEMENT     = V4 / '02b_llm_validation'  / 'agreement_metrics.parquet'
PATH_STOCKINESS    = V4 / '03_stockiness'       / 'stockiness_long.parquet'
PATH_TOP_OVERALL   = V4 / '03_stockiness'       / 'top_stock_by_genre_overall.parquet'
PATH_TOP_NOUNS     = V4 / '03_stockiness'       / 'top_stock_by_genre_nouns.parquet'
PATH_TOP_VERBS     = V4 / '03_stockiness'       / 'top_stock_by_genre_verbs.parquet'
PATH_TOP_CHUNKS    = V4 / '03_stockiness'       / 'top_stock_by_genre_chunks.parquet'
PATH_TOP_ENTITIES  = V4 / '03_stockiness'       / 'top_stock_by_genre_entities.parquet'
PATH_CLASSIFIED    = V4 / '03b_classification'  / 'element_classifications.parquet'
PATH_STK_CLASS     = V4 / '03b_classification'  / 'stockiness_long_classified.parquet'
PATH_CENTRALITY    = V4 / '03c_network'         / 'element_centrality_by_genre.parquet'
PATH_COMBINATIONS  = V4 / '03c_network'         / 'combination_stockiness.parquet'
PATH_SURPRISAL     = V4 / '04_surprisal'        / 'per_film_surprisal.parquet'
PATH_FILM_STOCK    = V4 / '04_surprisal'        / 'per_film_mean_stockiness.parquet'
PATH_CORR_TABLE    = V4 / '04_surprisal'        / 'correlation_table.parquet'
PATH_SEM_VALID     = V4 / '05_embeddings'       / 'semantic_validation.parquet'
PATH_FILT_VALID    = V4 / '05_embeddings'       / 'filter_validation.parquet'
PATH_EMB_INDEX     = V4 / '05_embeddings'       / 'element_embeddings_index.parquet'
PATH_EMB_NPY       = V4 / '05_embeddings'       / 'element_embeddings.npy'
PATH_CENTROIDS     = V4 / '05_embeddings'       / 'genre_centroids.parquet'
PATH_BY_BIN        = V4 / '08_temporal'         / 'stockiness_by_bin.parquet'
PATH_LIFECYCLE     = V4 / '08_temporal'         / 'element_lifecycle.parquet'
PATH_CHANGE_PTS    = V4 / '08_temporal'         / 'change_points.parquet'
PATH_SURVIVAL      = V4 / '08_temporal'         / 'survival_curves.parquet'

# Output tree
OUT_DIR    = V4 / '09_deliverables'
FIG_PDF    = OUT_DIR / 'figures' / 'pdf'
FIG_PNG    = OUT_DIR / 'figures' / 'png'
FIG_SRC    = OUT_DIR / 'figures' / 'source_data'
TAB_LATEX  = OUT_DIR / 'tables'  / 'latex'
TAB_CSV    = OUT_DIR / 'tables'  / 'csv'
for p in [FIG_PDF, FIG_PNG, FIG_SRC, TAB_LATEX, TAB_CSV]:
    p.mkdir(parents=True, exist_ok=True)

# Clean stale outputs from previous runs so the directory does not
# accumulate orphaned files when identifiers change between rebuilds.
for d in [FIG_PDF, FIG_PNG, FIG_SRC, TAB_LATEX, TAB_CSV]:
    for f in d.glob('*'):
        if f.is_file():
            f.unlink()

# --- Matplotlib defaults (serif body, vector-friendly) ----------------
plt.rcParams.update({
    'font.family'       : 'serif',
    'font.serif'        : ['Times New Roman', 'DejaVu Serif', 'serif'],
    'font.size'         : 11,
    'axes.titlesize'    : 11,
    'axes.labelsize'    : 11,
    'axes.linewidth'    : 0.8,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'xtick.labelsize'   : 9,
    'ytick.labelsize'   : 9,
    'legend.fontsize'   : 9,
    'figure.dpi'        : 100,
    'savefig.dpi'       : 300,
    'savefig.bbox'      : 'tight',
    'pdf.fonttype'      : 42,
    'ps.fonttype'       : 42,
})

# --- Canonical genres + 17-color palette (fixed across all figures) ---
CANONICAL_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Drama', 'Family', 'Fantasy', 'History', 'Horror',
    'Music', 'Mystery', 'Romance', 'Science Fiction',
    'Thriller', 'War', 'Western',
]
N_GENRES = len(CANONICAL_GENRES)
_tab20 = plt.get_cmap('tab20').colors
_idx = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 1, 3, 5, 7, 9, 11, 13]
GENRE_COLORS = {g: _tab20[i] for g, i in zip(CANONICAL_GENRES, _idx)}

# --- Franchise blocklist (post-canonicalization) ----------------------
# Updated for the post-canonicalization surface forms. Canonicalization
# in Notebook 02 changed some franchise terms (e.g., 'stooges' was
# singularized to 'stooge', which 03b then mis-classified as a generic
# character_type rather than the original character_name). This blocklist
# is the auditable post-hoc correction for the residual franchise leaks
# we have confirmed in the post-canon data. Each entry has a one-line
# rationale below.
FRANCHISE_BLOCKLIST = {
    # Three Stooges (Comedy)
    'stooge',            # post-canon singular of 'stooges'; 03b mis-tagged character_type
    'stooges',           # safety net; the entity_ORG form sometimes persists

    # Looney Tunes / MGM / Tex Avery cartoons (Animation / Family / Comedy)
    'panther',           # Pink Panther (peaks 1965, 1975 — Pink Panther films)
    'coyote',            # Wile E. Coyote (peaks Family 1955)
    'road runner',       # Looney Tunes (chunk + 5 entity types all leak)
    'roadrunner',        # alternate spelling
    'tasmanian devil',   # Looney Tunes (chunk + entity_ORG)
    'wabbit',             # 'wascally wabbit' Bugs reference
    'daffy',             # Daffy Duck
    'felix the cat',
    'speedy gonzales',
    'mickey mouse',
    'donald duck',

    # Star Trek (Science Fiction)
    'enterprise',
    'enterprise crew',
    'shuttlecraft',

    # Marvel / DC (Action)
    'spider man',
    'spider-man',
    'spiderman',
    'batman',
    'superman',
    'wonder woman',

    # Bond / Indy / Terminator / LOTR (Action / Adventure / Fantasy)
    'james bond',
    'indiana jones',
    'terminator',         # 03b mis-tagged narrative character_type
    'hobbit',             # Tolkien franchise
}

# --- Canonical-element lifecycle picks for F-VII-2 --------------------
# Eight doctrinally salient elements with verified sustained presence in
# stock_by_bin. Each was checked: ≥7 bins of stock duration, peak z > 2,
# and clear single-genre association. These are the body figure's
# illustrative lifecycle examples; the per-genre principled top-6 panels
# go in Appendix B (F-App-B18..B34).
LIFECYCLE_EXAMPLES = [
    ('sheriff',     'noun',  'Western'),          # 19b, peak 5.90
    ('vampire',     'noun',  'Horror'),           # 16b, peak 11.14
    ('dragon',      'noun',  'Fantasy'),          # 13b, peak 9.76
    ('soldier',     'noun',  'War'),              # 21b (full corpus), peak 12.06
    ('detective',   'noun',  'Crime'),            # 19b, peak 4.12
    ('spaceship',   'noun',  'Science Fiction'),  # 14b, peak 7.21
    ('wagon train', 'chunk', 'Western'),          # 7b, peak 3.93
    ('marriage',    'noun',  'Romance'),          # 19b, peak 11.25
]
LIFECYCLE_TOP_PER_GENRE = 6  # Appendix B per-genre panel count

# --- Run-configuration constants --------------------------------------
TOP_K_PER_TYPE       = 25     # Appendix per-type tables
TOP_K_OVERALL        = 10     # T-IV-1 body table
TOP_K_FILTER_PANEL   = 50     # F-IV-3 pre/post-filter composition
NETWORK_TOP_N        = 35     # F-VI-1 nodes shown
NETWORK_TOP_EDGES    = 120    # F-VI-1 max edges drawn
COMBO_TOP_PER_GENRE  = 5      # T-VI-1 top combinations
EMB_SAMPLE_N         = 8000   # Embedding scatter subsample (deferred to App B)
SCATTER_SAMPLE_N     = 15000  # F-VI-2 centrality scatter subsample
PARTIAL_SAMPLE_N     = 6000   # F-V-1 length-confound scatter subsample
LITIGATION_TOP_PER   = 20     # T-VIII-1 candidates per genre
LITIGATION_DURABILITY_MIN_BINS = 4    # T-VIII-1 min bins for durability
LITIGATION_PCT_THRESHOLD       = 0.95 # T-VIII-1 ensemble z percentile

ANCHOR_GENRE_FOR_NETWORK = 'Western'  # F-VI-1 body figure

# --- Worked-example case (Part VIII, T-VIII-2) ------------------------
# A constructed Western copyright dispute. Defendant moves for summary
# dismissal on scènes à faire grounds for these seven elements.
# Manuscript can cite this as a 'how the framework would resolve a
# motion' demonstration. Element list is hand-picked to span clearly
# stock (sheriff, ranch), borderline (gunslinger), and transient
# (wagon train).
WORKED_CASE_NAME = 'Hypothetical Western Copyright Dispute'
WORKED_CASE_GENRE = 'Western'
WORKED_CASE_ELEMENTS = [
    # Clearly stock (z >= 10, sustained, narrative)
    ('sheriff',     'noun'),
    ('outlaw',      'noun'),
    ('saloon',      'noun'),
    ('ranch',       'noun'),
    ('cowboy',      'noun'),
    # Moderate stockiness (z 5-10, narrative, durable)
    ('wagon train', 'chunk'),
    ('gunslinger',  'noun'),
    # Contrast: high stockiness but filtered (place name, not narrative)
    ('texas',       'entity_GPE'),
    # Contrast: filtered franchise term, low stockiness
    ('lone ranger', 'chunk'),
    # Contrast: narrative but insufficient stockiness in Western
    ('chair',       'noun'),
]

print('Setup complete.')
print(f'  ROOT             : {ROOT}')
print(f'  OUTPUT DIR       : {OUT_DIR}')
print(f'  blocklist size   : {len(FRANCHISE_BLOCKLIST)}')
print(f'  lifecycle picks  : {len(LIFECYCLE_EXAMPLES)}')
print(f'  anchor genre     : {ANCHOR_GENRE_FOR_NETWORK}')
print(f'  worked-case      : {WORKED_CASE_NAME} ({len(WORKED_CASE_ELEMENTS)} elements)')


  Google Drive already mounted; skipping remount.
Setup complete.
  ROOT             : /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational
  OUTPUT DIR       : /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/09_deliverables
  blocklist size   : 26
  lifecycle picks  : 8
  anchor genre     : Western
  worked-case      : Hypothetical Western Copyright Dispute (10 elements)


In [15]:
# ----------------------------------------------------------------------
# Load upstream parquets and apply the franchise blocklist
#
# Reads every input the notebook needs in one cell with shape
# assertions. If any file is missing or the wrong shape, this cell
# fails loudly so the downstream figure cells do not silently emit
# broken plots. Then applies the franchise blocklist to four tables:
#   - classifications (sets is_narrative_element=False)
#   - stockiness_classified (same)
#   - centrality (drops the row entirely)
#   - combinations (drops rows where either element is blocklisted)
#
# This makes the blocklist consistent across every downstream cell;
# subsequent cells just use the in-memory tables.
# ----------------------------------------------------------------------

print('Loading upstream parquets...\n')

# Notebook 01c
movies = pd.read_parquet(PATH_MOVIES)
assert len(movies) == 45958, f'movies_union has {len(movies)} rows; expected 45,958'
print(f'  movies                 {movies.shape}')

# Notebook 02
element_index = pd.read_parquet(PATH_ELEM_IDX)
print(f'  element_index          {element_index.shape}')

# Notebook 02b
agreement = pd.read_parquet(PATH_AGREEMENT)
print(f'  agreement              {agreement.shape}')

# Notebook 03
stockiness   = pd.read_parquet(PATH_STOCKINESS)
top_overall  = pd.read_parquet(PATH_TOP_OVERALL)
top_nouns    = pd.read_parquet(PATH_TOP_NOUNS)
top_verbs    = pd.read_parquet(PATH_TOP_VERBS)
top_chunks   = pd.read_parquet(PATH_TOP_CHUNKS)
top_entities = pd.read_parquet(PATH_TOP_ENTITIES)
print(f'  stockiness             {stockiness.shape}')
print(f'  top_overall            {top_overall.shape}')

# Notebook 03b
classifications        = pd.read_parquet(PATH_CLASSIFIED)
stockiness_classified  = pd.read_parquet(PATH_STK_CLASS)
print(f'  classifications        {classifications.shape}')
print(f'  stockiness_classified  {stockiness_classified.shape}')

# Apply blocklist to classifications + stockiness_classified
_bl = {s.lower() for s in FRANCHISE_BLOCKLIST}
_mask_cls = classifications['element'].astype(str).str.lower().isin(_bl)
_mask_stk = stockiness_classified['element'].astype(str).str.lower().isin(_bl)
n_cls_flipped = int(_mask_cls.sum())
n_stk_flipped = int(_mask_stk.sum())
classifications.loc[_mask_cls, 'is_narrative_element'] = False
classifications.loc[_mask_cls, 'element_taxonomy']    = 'character_name'
stockiness_classified.loc[_mask_stk, 'is_narrative_element'] = False
stockiness_classified.loc[_mask_stk, 'element_taxonomy']    = 'character_name'
print(f'  blocklist applied      forced {n_cls_flipped} classification rows '
      f'and {n_stk_flipped} per-cell rows to non-narrative')

# Notebook 03c
centrality   = pd.read_parquet(PATH_CENTRALITY)
combinations = pd.read_parquet(PATH_COMBINATIONS)
print(f'  centrality             {centrality.shape}')
print(f'  combinations           {combinations.shape}')

# Apply blocklist to centrality + combinations (drop rows outright)
_nc0 = len(centrality)
centrality = centrality[~centrality['element'].astype(str).str.lower().isin(_bl)].copy()
_nb0 = len(combinations)
combinations = combinations[
    ~combinations['element_i'].astype(str).str.lower().isin(_bl) &
    ~combinations['element_j'].astype(str).str.lower().isin(_bl)
].copy()
print(f'  blocklist dropped      {_nc0 - len(centrality)} centrality rows '
      f'and {_nb0 - len(combinations)} combination rows')

# Notebook 04
per_film_surprisal  = pd.read_parquet(PATH_SURPRISAL)
per_film_stockiness = pd.read_parquet(PATH_FILM_STOCK)
correlation_table   = pd.read_parquet(PATH_CORR_TABLE)
print(f'  per_film_surprisal     {per_film_surprisal.shape}')
print(f'  per_film_stockiness    {per_film_stockiness.shape}')
print(f'  correlation_table      {correlation_table.shape}')

# Notebook 05
sem_validation     = pd.read_parquet(PATH_SEM_VALID)
filt_validation    = pd.read_parquet(PATH_FILT_VALID)
elem_embedding_idx = pd.read_parquet(PATH_EMB_INDEX)
elem_embeddings    = np.load(PATH_EMB_NPY)
genre_centroids    = pd.read_parquet(PATH_CENTROIDS)
print(f'  sem_validation         {sem_validation.shape}')
print(f'  filt_validation        {filt_validation.shape}')
print(f'  elem_embeddings        {elem_embeddings.shape}')

# Notebook 08
stock_by_bin      = pd.read_parquet(PATH_BY_BIN)
element_lifecycle = pd.read_parquet(PATH_LIFECYCLE)
change_points     = pd.read_parquet(PATH_CHANGE_PTS)
survival          = pd.read_parquet(PATH_SURVIVAL)
print(f'  stock_by_bin           {stock_by_bin.shape}')
print(f'  element_lifecycle      {element_lifecycle.shape}')
print(f'  change_points          {change_points.shape}')
print(f'  survival               {survival.shape}')

print('\nAll inputs loaded.')


Loading upstream parquets...

  movies                 (45958, 13)
  element_index          (5658714, 3)
  agreement              (1500, 24)
  stockiness             (456159, 31)
  top_overall            (5950, 9)
  classifications        (40877, 11)
  stockiness_classified  (456159, 38)
  blocklist applied      forced 38 classification rows and 358 per-cell rows to non-narrative
  centrality             (120604, 11)
  combinations           (9224996, 18)
  blocklist dropped      59 centrality rows and 2481 combination rows
  per_film_surprisal     (45958, 4)
  per_film_stockiness    (45953, 4)
  correlation_table      (18, 8)
  sem_validation         (17, 9)
  filt_validation        (17, 9)
  elem_embeddings        (40877, 1024)
  stock_by_bin           (815444, 26)
  element_lifecycle      (143691, 12)
  change_points          (338, 7)
  survival               (108, 6)

All inputs loaded.


In [16]:
# ----------------------------------------------------------------------
# Helpers: figure / table writers, headline registry, caption seeds
#
# Every output goes through one of two writer functions so the manifest,
# source-data mirror, LaTeX / PDF outputs, and caption seeds stay in
# lockstep. Numbers cited in the manuscript are registered through
# headline() so the macro file is generated automatically at the end.
# ----------------------------------------------------------------------

MANIFEST_ROWS = []
HEADLINE = {}
CAPTION_SEEDS = {}  # {ident: caption_text}

_NUM_WORDS = {str(i): w for i, w in enumerate(
    ['Zero', 'One', 'Two', 'Three', 'Four',
     'Five', 'Six', 'Seven', 'Eight', 'Nine'])}


# Convert snake_case_with_digits -> CamelCaseWithWords for LaTeX macros.
# LaTeX disallows digits and underscores in macro names.
def to_macro_name(key):
    parts = key.split('_')
    out = []
    for p in parts:
        if not p:
            continue
        result = ''
        last_digit = False
        for c in p:
            if c.isdigit():
                result += _NUM_WORDS[c]
                last_digit = True
            else:
                if not result or last_digit:
                    result += c.upper()
                else:
                    result += c
                last_digit = False
        out.append(result)
    return ''.join(out)


# Register a named number for the manuscript's macro file.
def headline(key, value, fmt=None, description=''):
    HEADLINE[key] = {'value': value, 'fmt': fmt, 'description': description}


# Filename-safe slug for a genre ('Science Fiction' -> 'Science_Fiction')
def safe_slug(g):
    return str(g).replace(' ', '_').replace('/', '_')


# Internal: remove existing manifest entry with this id, then append.
# This makes re-running a cell idempotent — the manifest always
# reflects the most recent write per id, regardless of how many times
# a cell was re-executed during an interactive Colab session.
def _register_manifest(row):
    global MANIFEST_ROWS
    MANIFEST_ROWS = [r for r in MANIFEST_ROWS if r['id'] != row['id']]
    MANIFEST_ROWS.append(row)


# Save a figure as PDF + PNG; mirror source data; register caption seed.
def save_figure(fig, fig_id, source_data=None, source_notebook='',
                caption=''):
    pdf_path = FIG_PDF / f'{fig_id}.pdf'
    png_path = FIG_PNG / f'{fig_id}.png'
    fig.savefig(pdf_path)
    fig.savefig(png_path)
    src_rel = ''
    if source_data is not None:
        src_path = FIG_SRC / f'{fig_id}.csv'
        source_data.to_csv(src_path, index=False)
        src_rel = str(src_path.relative_to(OUT_DIR))
    plt.close(fig)
    _register_manifest({
        'kind'            : 'figure',
        'id'              : fig_id,
        'pdf_path'        : str(pdf_path.relative_to(OUT_DIR)),
        'png_path'        : str(png_path.relative_to(OUT_DIR)),
        'source_data'     : src_rel,
        'source_notebook' : source_notebook,
        'caption'         : caption,
    })
    if caption:
        CAPTION_SEEDS[fig_id] = caption
    print(f'  fig  {fig_id}')


# Save a dataframe as booktabs LaTeX + CSV; register caption seed.
def save_table(df, table_id, caption='', label=None, source_notebook='',
               column_format=None, escape=True, index=False,
               na_rep='--', float_format='%.3f'):
    if label is None:
        label = f'tab:{table_id}'
    latex_path = TAB_LATEX / f'{table_id}.tex'
    csv_path   = TAB_CSV   / f'{table_id}.csv'
    df.to_csv(csv_path, index=index)
    latex_str = df.to_latex(
        index=index, escape=escape, column_format=column_format,
        caption=caption, label=label, na_rep=na_rep,
        float_format=float_format, position='htbp',
    )
    with open(latex_path, 'w', encoding='utf-8') as f:
        f.write(latex_str)
    _register_manifest({
        'kind'            : 'table',
        'id'              : table_id,
        'pdf_path'        : '',
        'png_path'        : '',
        'source_data'     : str(csv_path.relative_to(OUT_DIR)),
        'source_notebook' : source_notebook,
        'caption'         : caption,
    })
    if caption:
        CAPTION_SEEDS[table_id] = caption
    print(f'  tab  {table_id}  ({len(df)} rows)')


# Convenience: a per-genre legend for figures using GENRE_COLORS.
def genre_legend_patches():
    return [mpatches.Patch(facecolor=GENRE_COLORS[g], edgecolor='none', label=g)
            for g in CANONICAL_GENRES]


print('Helpers ready.')
print(f'  outputs will land in {OUT_DIR}')


Helpers ready.
  outputs will land in /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/09_deliverables


In [17]:
# ----------------------------------------------------------------------
# Part II — Doctrinal background
#
# Outputs:
#   T-II-1   Restatement §18 descriptor → operationalization mapping
#
# The single doctrinal-to-empirical translation table. Each Restatement
# descriptor gets one row showing its operationalization, formula
# sketch, and the methodological source the framework draws on. The
# fifth descriptor ('customary') gets handled by Notebook 08's temporal
# recomputation rather than a per-cell measure in Notebook 03, which
# is why its formula column points to Notebook 08.
# ----------------------------------------------------------------------

print('Part II — Doctrinal background')

T_II_1 = pd.DataFrame([
    {'Descriptor (Restatement §18)': 'Stock',
     'Operationalization'    : 'Primary stockiness',
     'Formula'               : r'$P(e \mid g)\times P(g \mid e)$',
     'Source'                : r'Rosch (1975) cue validity'},
    {'Descriptor (Restatement §18)': 'Common (but distinctive)',
     'Operationalization'    : 'Smoothed TF-IDF',
     'Formula'               : r'$\textit{tf}\times[\log\frac{1+|G|}{1+k(e)}+1]$',
     'Source'                : 'sklearn smoothing'},
    {'Descriptor (Restatement §18)': 'Stereotypical',
     'Operationalization'    : r'Dunning $G^{2}$ (positive)',
     'Formula'               : r'$2\sum O\log(O/E)$',
     'Source'                : r'Dunning (1993); $\sim 2N \cdot KL$'},
    {'Descriptor (Restatement §18)': 'Standard',
     'Operationalization'    : 'Smoothed posterior',
     'Formula'               : r'$\frac{c(e,g)+\alpha P_{prior}(g)}{c(e)+\alpha}$',
     'Source'                : r'Dirichlet $\alpha=1$'},
    {'Descriptor (Restatement §18)': 'Customary',
     'Operationalization'    : 'Per-bin temporal recomputation',
     'Formula'               : r'Notebook 08, five-year bins ($\tau$)',
     'Source'                : 'Temporal lifecycle'},
])
save_table(
    T_II_1, 'T-II-1_doctrine_mapping',
    caption=(r'Mapping from Restatement of Copyright §18 descriptors '
             r'(``standard, stock, customary, stereotypical, or '
             r"common'') to the per-cell measures computed in "
             r'Notebook 03 and the temporal recomputation in '
             r'Notebook 08. Each descriptor receives a distinct '
             r'statistical operationalization rather than a synonymous '
             r'restatement, addressing the doctrinal critique that '
             r'judicial application treats the five descriptors '
             r'interchangeably.'),
    label='tab:doctrine', source_notebook='03_stockiness + 08_temporal',
    column_format='p{3cm}p{3cm}p{4cm}p{3cm}', escape=False,
)


Part II — Doctrinal background
  tab  T-II-1_doctrine_mapping  (5 rows)


In [18]:
# ----------------------------------------------------------------------
# Part III — Methodology
#
# Outputs:
#   T-III-1   Corpus composition (CMU + Kaggle union)
#   T-III-2   Per-element-type extraction summary
#   T-III-3   LLM validation metrics
#   F-III-1   Pipeline flow diagram
#
# Three small descriptive tables and a one-page pipeline diagram that
# the manuscript's Methodology section can use to describe the data
# and procedure without exhaustive prose recapitulation.
# ----------------------------------------------------------------------

print('Part III — Methodology')

# --- Headline numbers used across multiple sections -------------------
n_films            = len(movies)
n_unique_elements  = element_index.groupby(['element', 'element_type']).ngroups
n_cells_filtered   = len(stockiness)
n_elements_filt    = stockiness.groupby(['element', 'element_type']).ngroups
# The classifications cover the May 2026 vocabulary (40,877 pairs). Count only the pairs
# that are in the current filtered table (October 2026 filter correction).
_current_pairs = stockiness[['element', 'element_type']].drop_duplicates()
classifications_current = classifications.merge(_current_pairs, on=['element', 'element_type'])
n_narrative        = int(classifications_current['is_narrative_element'].sum())

headline('n_films_total',           n_films,           '{:,}',
         'Films in corpus union (CMU + Kaggle Wikipedia)')
headline('n_unique_elements_pre_filter', n_unique_elements, '{:,}',
         'Unique (element, type) pairs extracted before the 10/5 filter')
headline('n_cells_filtered',        n_cells_filtered,  '{:,}',
         'Cells in stockiness_long after the 10/5 filter')
headline('n_elements_filtered',     n_elements_filt,   '{:,}',
         'Unique (element, type) pairs after the 10/5 filter')
headline('n_narrative_elements',    n_narrative,       '{:,}',
         'Narrative (element, type) pairs after 03b classification + blocklist')

# --- T-III-1: corpus composition --------------------------------------
T_III_1 = pd.DataFrame([
    {'Source': 'CMU Movie Summary Corpus',
     'Films' : int(movies['cmu_wiki_id'].notna().sum())},
    {'Source': 'Kaggle Wikipedia Movie Plots',
     'Films' : int(movies['kaggle_wiki_id'].notna().sum())},
    {'Source': 'Overlap (films in both corpora)',
     'Films' : int((movies['cmu_wiki_id'].notna() & movies['kaggle_wiki_id'].notna()).sum())},
    {'Source': 'Union (deduplicated)',
     'Films' : n_films},
])
T_III_1['Films'] = T_III_1['Films'].astype(int)
save_table(
    T_III_1, 'T-III-1_corpus_composition',
    caption=('Composition of the 45,958-film corpus union. The CMU Movie '
             'Summary Corpus (Bamman et al., 2014) and the Kaggle '
             'Wikipedia Movie Plots dataset are unioned on Wikipedia '
             'page ID; the overlap is films present in both, '
             'deduplicated by union ID in the final table.'),
    label='tab:corpus', source_notebook='01c_union_corpus',
    column_format='lr', float_format='%d',
)

# --- T-III-2: per-type extraction summary -----------------------------
T_III_2 = (element_index.groupby('element_type')
           .agg(total_instances=('element', 'count'),
                unique_strings  =('element', 'nunique'))
           .reset_index())
T_III_2['mean_per_film'] = T_III_2['total_instances'] / n_films
T_III_2 = T_III_2.sort_values('total_instances', ascending=False)
T_III_2.columns = ['Element type', 'Total instances', 'Unique strings', 'Mean per film']
save_table(
    T_III_2, 'T-III-2_extraction_summary',
    caption=('Per-element-type extraction summary across the 45,958-film '
             'corpus, after the morph-gated canonicalization step in '
             'Notebook 02. Counts are before the 10/5 filter that '
             'drives the stockiness measures.'),
    label='tab:extraction', source_notebook='02_element_extraction',
    column_format='lrrr', float_format='%.2f',
)

# --- T-III-3: LLM validation metrics ----------------------------------
chunk_p     = float(agreement['chunk_precision'].mean())
chunk_r     = float(agreement['chunk_recall'].mean())
chunk_j     = float(agreement['chunk_jaccard'].mean())
pooled_r    = float(agreement['recall'].mean())
n_validated = len(agreement)
headline('chunk_precision',  chunk_p,  '{:.3f}', 'Mean chunk precision in 02b')
headline('chunk_recall',     chunk_r,  '{:.3f}', 'Mean chunk recall in 02b')
headline('pooled_recall',    pooled_r, '{:.3f}', 'Pooled recall (all element types) in 02b')
headline('n_validated_films',n_validated,'{:,}',
         'Films stratified-sampled for 02b validation (88 or 89 per genre, seed 20260219)')

T_III_3 = pd.DataFrame([
    {'Metric': 'Films validated (equal-stratified, 17 genres)',
     'Value' : f'{n_validated:,}'},
    {'Metric': 'Chunk precision (per-film mean): spaCy chunks reproduced by Sonnet',
     'Value' : f'{chunk_p:.3f}'},
    {'Metric': 'Chunk recall (per-film mean): LLM chunks captured by spaCy',
     'Value' : f'{chunk_r:.3f}'},
    {'Metric': 'Chunk Jaccard agreement (per-film mean)',
     'Value' : f'{chunk_j:.3f}'},
    {'Metric': 'Pooled recall (per-film mean across all element types)',
     'Value' : f'{pooled_r:.3f}'},
])
save_table(
    T_III_3, 'T-III-3_llm_validation',
    caption=(f'LLM-versus-spaCy agreement on a stratified sample of {n_validated:,} '
             'films (88 or 89 per genre, fixed seed 20260219). Metrics are '
             'per-film means. Chunk-level precision is the fair direct '
             'comparison because the LLM is not prompted to produce '
             'single-word nouns or bare verbs. Chunk recall is low '
             'because the LLM primarily produces abstract themes and '
             'entity descriptions; pooled recall across all element '
             'types (final row) is the broader measure of whether '
             'spaCy captures the conceptual surface the LLM identifies.'),
    label='tab:validation', source_notebook='02b_llm_validation',
    column_format='lr',
)

# --- F-III-1: pipeline flow diagram (matplotlib) ----------------------
# A vertical-flow diagram showing the data path through the eight
# upstream notebooks. Boxes are rounded rectangles; flow arrows are
# curved fancy-arrows. Layout is hand-tuned to fit the manuscript page.

def draw_box(ax, x, y, w, h, text, fc='#f0f0f0', ec='black', fontsize=8.5):
    box = FancyBboxPatch((x-w/2, y-h/2), w, h,
                         boxstyle='round,pad=0.02',
                         facecolor=fc, edgecolor=ec, linewidth=0.8)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center',
            fontsize=fontsize, wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    arrow = FancyArrowPatch((x1, y1), (x2, y2),
                            arrowstyle='->', mutation_scale=12,
                            color='black', linewidth=0.6,
                            connectionstyle='arc3,rad=0')
    ax.add_patch(arrow)

fig, ax = plt.subplots(figsize=(8.5, 11))
ax.set_xlim(0, 10)
ax.set_ylim(0, 14)
ax.set_aspect('equal')
ax.axis('off')

# Top row: data input
draw_box(ax, 5.0, 13.3, 4.5, 0.7, 'Movie plot corpus\n(CMU + Kaggle, 45,958 films)', fc='#dbeafe')

# Notebook 02
draw_arrow(ax, 5.0, 12.9, 5.0, 12.4)
draw_box(ax, 5.0, 12.1, 5.0, 0.7,
         'Notebook 02 — Element extraction (spaCy)\nnouns, verbs, chunks, entities + morph-canonicalization', fc='#fef3c7')

# Notebook 03
draw_arrow(ax, 5.0, 11.7, 5.0, 11.2)
draw_box(ax, 5.0, 10.9, 5.0, 0.7,
         'Notebook 03 — Stockiness ensemble\n4 measures (Primary, TF-IDF, Dunning G², Posterior)', fc='#fef3c7')

# Notebook 03b
draw_arrow(ax, 5.0, 10.5, 5.0, 10.0)
draw_box(ax, 5.0, 9.7, 5.0, 0.7,
         'Notebook 03b — Narrative classification (Sonnet 4.6)\n10-category taxonomy, 94.5% gold-set accuracy', fc='#fef3c7')

# Branch into four downstream validation+analysis tracks
draw_arrow(ax, 5.0, 9.3, 2.0, 8.4)
draw_arrow(ax, 5.0, 9.3, 4.0, 8.4)
draw_arrow(ax, 5.0, 9.3, 6.0, 8.4)
draw_arrow(ax, 5.0, 9.3, 8.0, 8.4)

# Four downstream notebooks
draw_box(ax, 2.0, 8.1, 1.7, 0.6, 'Notebook 03c\nCombinations\n+ Networks', fc='#dcfce7', fontsize=7.5)
draw_box(ax, 4.0, 8.1, 1.7, 0.6, 'Notebook 04\nPlot surprisal\n(Qwen2.5-7B)', fc='#dcfce7', fontsize=7.5)
draw_box(ax, 6.0, 8.1, 1.7, 0.6, 'Notebook 05\nEmbeddings\n(Voyage-4-large)', fc='#dcfce7', fontsize=7.5)
draw_box(ax, 8.0, 8.1, 1.7, 0.6, 'Notebook 08\nTemporal\nlifecycle', fc='#dcfce7', fontsize=7.5)

# Validation results
draw_arrow(ax, 2.0, 7.8, 2.0, 7.0)
draw_arrow(ax, 4.0, 7.8, 4.0, 7.0)
draw_arrow(ax, 6.0, 7.8, 6.0, 7.0)
draw_arrow(ax, 8.0, 7.8, 8.0, 7.0)

draw_box(ax, 2.0, 6.7, 1.7, 0.5, 'PMI-weighted\nedges, eigenvector\ncentrality', fc='#f3e8ff', fontsize=7)
draw_box(ax, 4.0, 6.7, 1.7, 0.5, 'Discriminant validity:\npartial r = −0.16\n(length-controlled)', fc='#f3e8ff', fontsize=7)
draw_box(ax, 6.0, 6.7, 1.7, 0.5, "Semantic validity:\nArm 1 d=0.68,\nArm 2 d=0.67", fc='#f3e8ff', fontsize=7)
draw_box(ax, 8.0, 6.7, 1.7, 0.5, 'Survival, lifecycle,\nchange-points,\nemerging conventions', fc='#f3e8ff', fontsize=7)

# Converge into a single "framework outputs" layer
draw_arrow(ax, 2.0, 6.4, 5.0, 5.5)
draw_arrow(ax, 4.0, 6.4, 5.0, 5.5)
draw_arrow(ax, 6.0, 6.4, 5.0, 5.5)
draw_arrow(ax, 8.0, 6.4, 5.0, 5.5)

draw_box(ax, 5.0, 5.2, 7.5, 0.8,
         'Notebook 09 — Manuscript deliverables\n'
         '(this notebook: tables, figures, headline numbers, caption seeds)',
         fc='#ffe4e6')

# Output to manuscript layers
draw_arrow(ax, 5.0, 4.8, 2.5, 4.0)
draw_arrow(ax, 5.0, 4.8, 5.0, 4.0)
draw_arrow(ax, 5.0, 4.8, 7.5, 4.0)

draw_box(ax, 2.5, 3.7, 2.5, 0.5, 'Headline numbers\n(headline_numbers.tex)', fc='#fef9c3', fontsize=7.5)
draw_box(ax, 5.0, 3.7, 2.5, 0.5, 'Body figures + tables\n(figures/, tables/)', fc='#fef9c3', fontsize=7.5)
draw_box(ax, 7.5, 3.7, 2.5, 0.5, 'Appendix:\nper-genre fan-outs', fc='#fef9c3', fontsize=7.5)

# Final: Overleaf manuscript
draw_arrow(ax, 5.0, 3.4, 5.0, 2.7)
draw_box(ax, 5.0, 2.4, 4.5, 0.7, 'Overleaf manuscript\n(law-review article on scènes à faire)', fc='#dbeafe')

# Legend
ax.text(0.3, 1.2, 'Notebook stages (yellow)', fontsize=7.5, va='center')
draw_box(ax, 2.0, 1.2, 0.3, 0.18, '', fc='#fef3c7')
ax.text(2.4, 1.2, 'Validation tracks (green)', fontsize=7.5, va='center')
draw_box(ax, 4.4, 1.2, 0.3, 0.18, '', fc='#dcfce7')
ax.text(4.8, 1.2, 'Findings (purple)', fontsize=7.5, va='center')
draw_box(ax, 6.5, 1.2, 0.3, 0.18, '', fc='#f3e8ff')
ax.text(6.9, 1.2, 'Manuscript outputs (cream)', fontsize=7.5, va='center')

plt.title('Empirical computational scènes à faire framework — pipeline flow',
          fontsize=11, pad=8)
plt.tight_layout()
save_figure(
    fig, 'F-III-1_pipeline_flow', source_data=None,
    source_notebook='09_deliverables (this notebook)',
    caption=('Pipeline flow for the empirical computational scènes à faire '
             'framework. Notebook 02 extracts narrative elements from the '
             'corpus; Notebook 03 computes the four-measure stockiness '
             'ensemble; Notebook 03b classifies elements as narrative or '
             'filtered. Four downstream tracks (03c combinations, 04 '
             'surprisal discriminant validity, 05 semantic validity, 08 '
             'temporal lifecycle) provide convergent validation and '
             'analyses that converge in Notebook 09 (this notebook) into '
             'the manuscript-ready outputs.'),
)


Part III — Methodology
  tab  T-III-1_corpus_composition  (4 rows)
  tab  T-III-2_extraction_summary  (11 rows)
  tab  T-III-3_llm_validation  (5 rows)
  fig  F-III-1_pipeline_flow


In [19]:
# ----------------------------------------------------------------------
# Part IV — Core results
#
# Outputs:
#   T-IV-1   Top-10 narrative elements per genre (HEADLINE)
#   T-IV-2   Ten-category taxonomy with counts
#   F-IV-1   Within-genre measure-correlation heatmap
#   F-IV-2   Within-genre ensemble distributions (boxplots)
#   F-IV-3   Pre/post-filter top-K composition
#
# T-IV-1 is the framework's central empirical product — the table a
# court could read. F-IV-1 shows the four ensemble measures are
# complementary, not redundant. F-IV-2 shows the typical right-skew of
# the within-genre ensemble distribution. F-IV-3 shows what the
# narrative filter removes (named entities, copyrighted properties).
# ----------------------------------------------------------------------

print('Part IV — Core results')

# --- T-IV-1: top-10 narrative elements per genre ----------------------
# Uses stockiness_long_classified filtered to is_narrative_element=True.
# Dedup by surface form within genre to avoid showing 'murder' twice
# (noun + verb) in the body table.
narrative_only = stockiness_classified[
    stockiness_classified['is_narrative_element']
].copy()
T_IV_1_long = (narrative_only.sort_values(['genre', 'ensemble'],
                                          ascending=[True, False])
               .drop_duplicates(['genre', 'element'])
               .groupby('genre').head(TOP_K_OVERALL).copy())
T_IV_1_long['rank'] = T_IV_1_long.groupby('genre').cumcount() + 1
T_IV_1_wide = (T_IV_1_long.pivot_table(index='rank', columns='genre',
                                       values='element', aggfunc='first')
               .reindex(columns=CANONICAL_GENRES)
               .reset_index()
               .rename(columns={'rank': 'Rank'}))
save_table(
    T_IV_1_wide, 'T-IV-1_top10_per_genre',
    caption=('Top 10 narrative-classified stock elements per genre by the '
             'ensemble z-score. Rankings combine primary stockiness, '
             r'smoothed TF-IDF, Dunning $G^{2}$, and the Dirichlet-smoothed '
             'posterior, equal-weighted within-genre z-scores. Elements '
             'classified as named entities, copyrighted properties, or '
             'discourse markers by the Notebook 03b classifier are '
             'excluded. The franchise blocklist additionally removes '
             'a small set of unambiguous franchise terms (e.g., stooge, '
             'panther, coyote) that the classifier mis-tagged.'),
    label='tab:top10', source_notebook='03_stockiness + 03b_classification',
    column_format='l' + 'l' * N_GENRES, escape=True,
)

# --- T-IV-2: 10-category taxonomy with counts -------------------------
tax = (classifications_current.groupby('element_taxonomy')
       .size().rename('n').reset_index())
narr_cats = set(classifications_current.loc[
    classifications_current['is_narrative_element'], 'element_taxonomy'].unique())
tax['Role'] = tax['element_taxonomy'].apply(
    lambda x: 'Narrative' if x in narr_cats else 'Filtered')
tax = tax.sort_values(['Role', 'n'], ascending=[True, False]).reset_index(drop=True)
tax['share'] = tax['n'] / tax['n'].sum()
T_IV_2 = tax.rename(columns={'element_taxonomy': 'Category',
                             'n': 'Count', 'share': 'Share'})
T_IV_2 = T_IV_2[['Role', 'Category', 'Count', 'Share']]
save_table(
    T_IV_2, 'T-IV-2_taxonomy',
    caption=('Ten-category taxonomy applied by the Notebook 03b two-tier '
             f'Sonnet classifier across all {len(classifications_current):,} unique (element, type) '
             'pairs in the post-filter corpus. Narrative categories are '
             'kept for downstream analysis; filtered categories are '
             'excluded because they carry no doctrinal weight (named '
             'entities, copyrighted properties, generic discourse markers, '
             'place names that operate as identifiers rather than '
             'narrative elements).'),
    label='tab:taxonomy', source_notebook='03b_element_classification',
    column_format='llrr', float_format='%.3f',
)
narrative_share = float(classifications_current['is_narrative_element'].mean())
headline('narrative_share', narrative_share, '{:.1%}',
         'Share of unique (element, type) pairs classified as narrative')

# --- F-IV-1: within-genre measure-correlation heatmap -----------------
z_cols = ['primary_stockiness_z', 'tfidf_z', 'dunning_g2_positive_z', 'posterior_smoothed_z']
z_labels = ['Primary', 'Smoothed TF-IDF', r'Dunning $G^{2}$', 'Smoothed posterior']

per_genre_corr = []
for g, sub in stockiness.groupby('genre'):
    per_genre_corr.append(sub[z_cols].corr().values)
corr_mean = np.mean(per_genre_corr, axis=0)

fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(corr_mean, cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(z_labels, rotation=30, ha='right')
ax.set_yticklabels(z_labels)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{corr_mean[i, j]:.2f}',
                ha='center', va='center',
                color='white' if abs(corr_mean[i, j]) > 0.55 else 'black',
                fontsize=10)
cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label('Pearson $r$ (mean across 17 genres)')
ax.set_title('Within-genre correlation among ensemble members')
plt.tight_layout()

src = pd.DataFrame(corr_mean, index=z_labels, columns=z_labels).reset_index()
src.columns = ['measure'] + z_labels
save_figure(
    fig, 'F-IV-1_measure_correlation', source_data=src,
    source_notebook='03_stockiness',
    caption=('Mean within-genre Pearson correlations among the four '
             'ensemble z-scores, averaged across the seventeen canonical '
             'genres. Off-diagonal values are positive and substantial but '
             'not collinear; the four measures share signal but each '
             'captures structure the others do not. The smoothed posterior '
             'is essentially uncorrelated with the smoothed TF-IDF '
             '($r \\approx 0$), reflecting the design choice to make each '
             'measure operationalize a distinct Restatement §18 '
             'descriptor.'),
)
off_diag = corr_mean[np.triu_indices(4, k=1)]
headline('mean_measure_correlation', float(np.mean(off_diag)), '{:.2f}',
         'Mean off-diagonal correlation among the four ensemble z-scores')

# --- F-IV-2: within-genre ensemble distributions ----------------------
fig, ax = plt.subplots(figsize=(8.5, 4.6))
data = [stockiness[stockiness['genre'] == g]['ensemble'].values for g in CANONICAL_GENRES]
bp = ax.boxplot(
    data, labels=CANONICAL_GENRES, patch_artist=True, showfliers=False,
    whis=(5, 95), widths=0.65, medianprops={'color': 'black'},
)
for patch, g in zip(bp['boxes'], CANONICAL_GENRES):
    patch.set_facecolor(GENRE_COLORS[g])
    patch.set_edgecolor('black')
    patch.set_linewidth(0.6)
ax.axhline(0, color='gray', linewidth=0.6, linestyle='--')
ax.set_ylabel('Ensemble z-score')
ax.set_title('Within-genre ensemble distributions (5th-95th percentile whiskers)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()

summary = pd.DataFrame({
    g: pd.Series(d).describe() for g, d in zip(CANONICAL_GENRES, data)
}).T.reset_index().rename(columns={'index': 'genre'})
save_figure(
    fig, 'F-IV-2_ensemble_by_genre', source_data=summary,
    source_notebook='03_stockiness',
    caption=('Within-genre distributions of the ensemble z-score for the '
             r'seventeen canonical genres. Right-skewed; Dunning $G^{2}$ '
             'contributes the long upper tail. Boxes mark the interquartile '
             'range, whiskers the fifth and ninety-fifth percentiles, '
             'outliers suppressed for visual clarity. The right-skew is the '
             'doctrinal substrate of scènes à faire: a few extreme stock '
             'conventions sit above a long tail of moderately stock '
             'elements.'),
)

# --- F-IV-3: pre/post-filter top-K composition ------------------------
K = TOP_K_FILTER_PANEL
narrative_pairs = set(zip(
    classifications.loc[classifications['is_narrative_element'], 'element'],
    classifications.loc[classifications['is_narrative_element'], 'element_type'],
))

records = []
for g in CANONICAL_GENRES:
    sub = stockiness[stockiness['genre'] == g].sort_values('ensemble', ascending=False)
    pre = sub.head(K)
    mask = sub[['element', 'element_type']].apply(tuple, axis=1).isin(narrative_pairs)
    post = sub[mask].head(K)
    for stage, df_ in [('Pre-filter', pre), ('Post-filter', post)]:
        for t, cnt in df_['element_type'].value_counts().items():
            records.append({'genre': g, 'stage': stage,
                            'element_type': t, 'count': int(cnt)})
comp = pd.DataFrame(records)

present_types = comp['element_type'].unique().tolist()
canonical_order = ['noun', 'verb', 'chunk',
                   'entity_PERSON', 'entity_ORG', 'entity_LOC', 'entity_GPE',
                   'entity_NORP', 'entity_PRODUCT', 'entity_WORK_OF_ART',
                   'entity_EVENT', 'entity_FAC', 'entity_LANGUAGE',
                   'entity_LAW', 'entity_DATE', 'entity_TIME', 'entity_MONEY',
                   'entity_PERCENT', 'entity_QUANTITY',
                   'entity_ORDINAL', 'entity_CARDINAL']
type_order = ([t for t in canonical_order if t in present_types] +
              [t for t in present_types if t not in canonical_order])
type_palette = plt.get_cmap('Set3').colors
type_to_color = {t: type_palette[i % len(type_palette)]
                 for i, t in enumerate(type_order)}

fig, axes = plt.subplots(1, 2, figsize=(10, 5.5), sharey=True)
for ax, stage in zip(axes, ['Pre-filter', 'Post-filter']):
    sub = comp[comp['stage'] == stage]
    bottoms = {g: 0 for g in CANONICAL_GENRES}
    for t in type_order:
        vals = [int(sub[(sub['genre'] == g) &
                        (sub['element_type'] == t)]['count'].sum())
                for g in CANONICAL_GENRES]
        ax.bar(CANONICAL_GENRES, vals,
               bottom=[bottoms[g] for g in CANONICAL_GENRES],
               label=t, color=type_to_color[t], edgecolor='none', width=0.7)
        for i, g in enumerate(CANONICAL_GENRES):
            bottoms[g] += vals[i]
    ax.set_title(f'{stage} top {K}')
    ax.set_ylabel(f'Count in top {K}')
    plt.setp(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7,
               title='Element type', frameon=False)
plt.tight_layout()

save_figure(
    fig, 'F-IV-3_topk_composition', source_data=comp,
    source_notebook='03b_element_classification',
    caption=(f'Element-type composition of the top-{K} ranked elements in '
             f'each genre before (left) and after (right) the Notebook 03b '
             f'narrative filter. Filtering removes entity classes that '
             f'carry no doctrinal weight (named persons, organizations, '
             f'product brands, copyrighted properties), leaving the '
             f'common-noun, verb, and noun-chunk categories that '
             f'constitute the doctrinal substrate of scènes à faire.'),
)


Part IV — Core results
  tab  T-IV-1_top10_per_genre  (10 rows)
  tab  T-IV-2_taxonomy  (10 rows)
  fig  F-IV-1_measure_correlation


/tmp/ipykernel_4956/3912638630.py:128: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


  fig  F-IV-2_ensemble_by_genre
  fig  F-IV-3_topk_composition


In [20]:
# ----------------------------------------------------------------------
# Part V — Validation
#
# Outputs:
#   T-V-1   Per-genre partial r (discriminant validity, surprisal)
#   T-V-2   Per-genre Cohen's d (semantic Arm 1, top vs random)
#   T-V-3   Per-genre Cohen's d (Arm 2, narrative vs filtered)
#   F-V-1   Length-confound 2-panel (raw vs partial scatter)
#   F-V-2   Arm 1 Cohen's d per genre (horizontal bars)
#   F-V-3   Arm 2 Cohen's d per genre (horizontal bars)
#
# Three convergent validation arms:
#  1. Discriminant validity (Notebook 04): the framework is not a
#     re-skin of plot-level predictability. Partial r controlling for
#     log word count = -0.16, in the doctrinally expected direction
#     (predictable plots use stock elements) but small enough to
#     preserve the framework's independent doctrinal claim.
#  2. Convergent semantic validity, Arm 1 (Notebook 05): the framework's
#     count-based rankings align with an independent semantic
#     representation. Mean Cohen's d 0.68 across 17 genres.
#  3. 03b classification validity, Arm 2 (Notebook 05): the narrative
#     filter is doing real semantic work, not just statistical work.
#     16 of 17 genres confirm narrative > filtered.
# ----------------------------------------------------------------------

print('Part V — Validation')

# --- T-V-1: per-genre partial correlations ----------------------------
T_V_1 = correlation_table[correlation_table['genre'].isin(CANONICAL_GENRES)].copy()
T_V_1 = T_V_1.sort_values('partial_pearson_r')
T_V_1 = T_V_1.rename(columns={
    'genre'             : 'Genre',
    'n_films'           : 'n',
    'raw_pearson_r'     : r'Raw $r$',
    'partial_pearson_r' : r'Partial $r$',
    'raw_spearman_rho'  : r'Raw $\rho$',
})
T_V_1 = T_V_1[['Genre', 'n', r'Raw $r$', r'Partial $r$', r'Raw $\rho$']]
save_table(
    T_V_1, 'T-V-1_partial_correlation',
    caption=('Per-genre Pearson correlations between mean plot-level '
             'surprisal (Qwen2.5-7B) and mean per-film narrative '
             r'stockiness, raw and partial (controlling for $\log$ word '
             r'count). All seventeen genres show negative partial '
             'correlations, in the doctrinally expected direction: more '
             'predictable plots use more stock elements. The corpus-wide '
             'partial $r = -0.164$ sits in the "weak/no relationship" '
             '(|r| < 0.20) band, confirming that the framework\'s '
             'element-level measures are not a re-skin of plot-level '
             'language-model predictability.'),
    label='tab:partialr', source_notebook='04_plot_surprisal',
    column_format='lrrrr', escape=False, float_format='%.3f',
)
n_neg = int((T_V_1[r'Partial $r$'] < 0).sum())
headline('n_genres_negative_partial_r', n_neg, '{:d}',
         'Number of canonical genres with negative partial Pearson r (of 17)')

# --- T-V-2: per-genre Arm 1 Cohen's d ---------------------------------
T_V_2 = sem_validation.copy().sort_values('cohens_d', ascending=False)
T_V_2 = T_V_2.rename(columns={
    'genre'            : 'Genre',
    'n_top'            : 'n_top',
    'n_rand'           : 'n_rand',
    'top50_mean_cosine': r'Top-50 cosine',
    'random50_mean_cosine': r'Random-50 cosine',
    'gap'              : 'Gap',
    'cohens_d'         : r"Cohen's $d$",
    't_stat'           : r'$t$',
    'p_value'          : r'$p$',
})
save_table(
    T_V_2, 'T-V-2_arm1_cohens_d',
    caption=('Convergent semantic validity, Arm 1 (Notebook 05). For each '
             'genre, the top 50 stock elements by ensemble z and a random '
             'sample of 50 elements are projected to the genre centroid '
             "in Voyage-4-large embedding space. Cohen's $d$ measures the "
             'mean-cosine gap between the two pools. A positive gap '
             "confirms the framework's count-based ranking aligns with an "
             'independent semantic representation. Genres sorted descending '
             "by Cohen's $d$."),
    label='tab:arm1', source_notebook='05_embeddings_and_topics',
    column_format='lrrrrrrrr', escape=False, float_format='%.3f',
)
mean_d_arm1 = float(sem_validation['cohens_d'].mean())
median_d_arm1 = float(sem_validation['cohens_d'].median())
n_arm1_sig = int((sem_validation['p_value'] < 0.05).sum())
n_arm1_large = int((sem_validation['cohens_d'] >= 0.8).sum())
headline('cohens_d_mean_arm_1',   mean_d_arm1,   '{:.2f}',
         "Mean Cohen's d across 17 genres in Arm 1")
headline('cohens_d_median_arm_1', median_d_arm1, '{:.2f}',
         "Median Cohen's d across 17 genres in Arm 1")
headline('n_genres_arm_1_sig', n_arm1_sig, '{:d}',
         'Arm 1 genres significant at p<0.05 (of 17)')
headline('n_genres_arm_1_large', n_arm1_large, '{:d}',
         'Arm 1 genres with large effect d>=0.8 (of 17)')

# --- T-V-3: per-genre Arm 2 Cohen's d ---------------------------------
T_V_3 = filt_validation.copy().sort_values('cohens_d', ascending=False)
T_V_3 = T_V_3.rename(columns={
    'genre'           : 'Genre',
    'n_narr'          : r'$n_{narr}$',
    'n_filt'          : r'$n_{filt}$',
    'top_narr_mean'   : 'Narr. cosine',
    'top_filt_mean'   : 'Filt. cosine',
    'narr_minus_filt' : 'Gap',
    'cohens_d'        : r"Cohen's $d$",
    't_stat'          : r'$t$',
    'p_value'         : r'$p$',
})
save_table(
    T_V_3, 'T-V-3_arm2_cohens_d',
    caption=('Convergent semantic validity, Arm 2 (Notebook 05). For each '
             'genre, the top 50 narrative-classified and top 50 filtered '
             'elements are projected to the genre centroid in '
             "Voyage-4-large embedding space. Cohen's $d$ measures the "
             'mean-cosine gap. A positive gap confirms that Notebook '
             "03b's narrative-vs-filtered classification is doing real "
             'semantic work, not just statistical work. 16 of 17 genres '
             'show narrative > filtered; Animation is the documented '
             "anomaly (see Discussion, Part IX)."),
    label='tab:arm2', source_notebook='05_embeddings_and_topics',
    column_format='lrrrrrrrr', escape=False, float_format='%.3f',
)
mean_d_arm2 = float(filt_validation['cohens_d'].mean())
n_arm2_narr_greater = int((filt_validation['cohens_d'] > 0).sum())
headline('cohens_d_mean_arm_2', mean_d_arm2, '{:.2f}',
         "Mean Cohen's d across 17 genres in Arm 2")
headline('n_genres_arm_2_narr_greater', n_arm2_narr_greater, '{:d}',
         'Arm 2 genres where narrative > filtered (of 17)')

# --- F-V-1: length-confound 2-panel -----------------------------------
joined = (per_film_surprisal[['union_id', 'mean_surprisal']]
          .merge(per_film_stockiness[['union_id', 'mean_stockiness_overall']],
                 on='union_id', how='inner')
          .merge(movies[['union_id', 'word_count']],
                 on='union_id', how='inner'))
joined = joined[joined['word_count'] > 0].copy()
joined['log_wc'] = np.log(joined['word_count'])

raw_r = float(joined['mean_surprisal'].corr(joined['mean_stockiness_overall']))

def partial_corr(df, x, y, z):
    rxy = df[x].corr(df[y])
    rxz = df[x].corr(df[z])
    ryz = df[y].corr(df[z])
    return float((rxy - rxz * ryz) / np.sqrt((1 - rxz ** 2) * (1 - ryz ** 2)))

partial_r = partial_corr(joined, 'mean_surprisal',
                         'mean_stockiness_overall', 'log_wc')
headline('partial_r_corpuswide', partial_r, '{:+.3f}',
         'Partial Pearson r between surprisal and stockiness, controlling for log word count')
headline('raw_r_corpuswide', raw_r, '{:+.3f}',
         'Raw Pearson r between surprisal and stockiness (length-confounded)')

def residualize(y, x):
    slope, intercept = np.polyfit(x, y, 1)
    return y - (slope * x + intercept)

joined['surp_resid']  = residualize(joined['mean_surprisal'].values, joined['log_wc'].values)
joined['stock_resid'] = residualize(joined['mean_stockiness_overall'].values, joined['log_wc'].values)

sub_raw   = joined.sample(n=min(PARTIAL_SAMPLE_N, len(joined)), random_state=42)
sub_resid = joined.sample(n=min(PARTIAL_SAMPLE_N, len(joined)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
ax = axes[0]
ax.scatter(sub_raw['mean_surprisal'], sub_raw['mean_stockiness_overall'],
           s=3, color='#444444', alpha=0.3, edgecolors='none')
ax.set_xlabel('Mean per-token surprisal (nats)')
ax.set_ylabel('Mean per-film stockiness')
ax.set_title(f'Raw: $r$ = {raw_r:+.3f}')

ax = axes[1]
ax.scatter(sub_resid['surp_resid'], sub_resid['stock_resid'],
           s=3, color='#4477AA', alpha=0.3, edgecolors='none')
slope, intercept = np.polyfit(sub_resid['surp_resid'],
                              sub_resid['stock_resid'], 1)
xx = np.linspace(sub_resid['surp_resid'].min(),
                 sub_resid['surp_resid'].max(), 100)
ax.plot(xx, slope * xx + intercept, color='black', linewidth=1.2)
ax.set_xlabel('Surprisal residual (after log word count)')
ax.set_ylabel('Stockiness residual (after log word count)')
ax.set_title(f'Partial: $r$ = {partial_r:+.3f}')

plt.suptitle('Surprisal vs stockiness, raw and length-controlled', y=1.03)
plt.tight_layout()

src = joined[['union_id', 'mean_surprisal', 'mean_stockiness_overall',
              'log_wc', 'surp_resid', 'stock_resid']]
save_figure(
    fig, 'F-V-1_length_confound', source_data=src,
    source_notebook='04_plot_surprisal',
    caption=('Mean plot-level surprisal (Qwen2.5-7B) versus mean per-film '
             'stockiness. The raw scatter (left) shows near-zero '
             'correlation because both measures correlate with log word '
             'count with opposite signs: longer plots have lower per-token '
             "surprisal (the LM gets more context) and slightly lower "
             'per-film stockiness (longer plots include more idiosyncratic '
             'detail). The partial regression (right), controlling for '
             'log word count, reveals the doctrinally expected negative '
             "association: predictable plots use stock elements. The "
             "partial $|r| = 0.16$ is below the 0.20 'weak/no relationship' "
             "threshold, confirming the framework's element-level "
             'measures are not a re-skin of plot-level predictability.'),
)

# --- Helper for Cohen's d bar charts ----------------------------------
def cohen_d_barchart(df, value_col, title, fig_id, source_notebook, caption):
    df_sorted = df.sort_values(value_col, ascending=True).copy()
    fig, ax = plt.subplots(figsize=(7, 5.5))
    bars = ax.barh(df_sorted['genre'], df_sorted[value_col],
                   color=[GENRE_COLORS[g] for g in df_sorted['genre']],
                   edgecolor='black', linewidth=0.5)
    for bar, p in zip(bars, df_sorted['p_value']):
        if p < 0.001: star = '***'
        elif p < 0.01: star = '**'
        elif p < 0.05: star = '*'
        else:          star = ''
        x = bar.get_width()
        align  = 'left' if x >= 0 else 'right'
        offset = 0.02 if x >= 0 else -0.02
        ax.text(x + offset, bar.get_y() + bar.get_height() / 2,
                f'{x:+.2f}{star}', va='center', ha=align, fontsize=8)
    ax.axvline(0, color='black', linewidth=0.6)
    for ref in (0.2, 0.5, 0.8):
        ax.axvline(ref, color='gray', linestyle=':', linewidth=0.5)
    ax.set_xlabel("Cohen's $d$")
    ax.set_title(title)
    plt.tight_layout()
    save_figure(fig, fig_id, source_data=df_sorted,
                source_notebook=source_notebook, caption=caption)

# --- F-V-2: Arm 1 Cohen's d bars --------------------------------------
cohen_d_barchart(
    sem_validation, 'cohens_d',
    title='Arm 1: top-50 stock vs random-50 (cosine to genre centroid)',
    fig_id='F-V-2_arm1_cohens_d',
    source_notebook='05_embeddings_and_topics',
    caption=("Convergent semantic validity, Arm 1. Cohen's $d$ per genre "
             "for the cosine-similarity gap between the top-50 stockiest "
             "and 50 random elements against the Voyage-4-large genre "
             "centroid. Significance asterisks: *** $p < 0.001$, "
             "** $p < 0.01$, * $p < 0.05$. Reference lines at the "
             "conventional effect-size thresholds $d = 0.2, 0.5, 0.8$. "
             "Mean $d = 0.68$ across the seventeen genres; 13 of 17 reach "
             "$p < 0.05$ and 7 of 17 reach the large-effect threshold "
             "$d \\geq 0.8$."),
)

# --- F-V-3: Arm 2 Cohen's d bars --------------------------------------
cohen_d_barchart(
    filt_validation, 'cohens_d',
    title='Arm 2: top-50 narrative vs top-50 filtered (cosine to genre centroid)',
    fig_id='F-V-3_arm2_cohens_d',
    source_notebook='05_embeddings_and_topics',
    caption=("03b classification validity (Arm 2). Cohen's $d$ per genre "
             "for the cosine-similarity gap between the top-50 narrative-"
             "classified and top-50 filtered elements against the genre "
             "centroid. Positive values confirm 03b's narrative-vs-"
             "filtered classification is doing real semantic work. "
             "Sixteen of seventeen genres show narrative > filtered; "
             "Animation is the documented anomaly (see Part IX), where "
             "the genre's semantic identity is partly carried by "
             "franchise characters that 03b classifies as filtered."),
)


Part V — Validation
  tab  T-V-1_partial_correlation  (17 rows)
  tab  T-V-2_arm1_cohens_d  (17 rows)
  tab  T-V-3_arm2_cohens_d  (17 rows)
  fig  F-V-1_length_confound
  fig  F-V-2_arm1_cohens_d
  fig  F-V-3_arm2_cohens_d


In [21]:
# ----------------------------------------------------------------------
# Part VI — Combinations and co-occurrence networks
#
# Outputs:
#   T-VI-1   Top-5 element combinations per genre (balanced + dedup)
#   F-VI-1   Western co-occurrence network (body figure)
#   F-VI-2   Centrality vs stockiness scatter
#
# Operationalizes the Indiana Jones doctrine: ordinary tropes that
# become identifying when stacked. T-VI-1 applies two paper-time
# refinements (balanced + substring exclusion + surface-form dedup)
# to filter out artifacts. F-VI-1 shows the Western network as the
# anchor body figure; remaining 16 per-genre networks live in
# Appendix B. F-VI-2 shows centrality and per-cell stockiness are
# correlated but not redundant.
# ----------------------------------------------------------------------

print('Part VI — Combinations and networks')

# --- T-VI-1: top-5 combinations per genre -----------------------------
# Apply three refinements:
#  (1) Balanced: require both elements at z >= 2 in genre
#  (2) Substring exclusion: drop tautological pairs (a contains b)
#  (3) Surface-form dedup: keep highest per (genre, element_i, element_j)
balanced = combinations[
    (combinations['z_ensemble_i'] >= 2.0) &
    (combinations['z_ensemble_j'] >= 2.0)
].copy()

def _no_substring(r):
    a = str(r['element_i']).lower()
    b = str(r['element_j']).lower()
    return (a not in b) and (b not in a)

balanced = balanced[balanced.apply(_no_substring, axis=1)]

combo_top = (balanced.sort_values(['genre', 'combination_stock'],
                                  ascending=[True, False])
             .drop_duplicates(['genre', 'element_i', 'element_j'])
             .groupby('genre').head(COMBO_TOP_PER_GENRE).reset_index(drop=True))
combo_top['Combination'] = combo_top.apply(
    lambda r: f"{r['element_i']} + {r['element_j']}", axis=1)
T_VI_1 = (combo_top[['genre', 'Combination',
                     'combination_stock', 'pmi_positive', 'lift']]
          .rename(columns={'genre'              : 'Genre',
                           'combination_stock'  : 'Combo stock',
                           'pmi_positive'       : 'PMI',
                           'lift'               : 'Lift'}))
save_table(
    T_VI_1, 'T-VI-1_top_combinations',
    caption=(r'Top five element combinations per genre by combination '
             r'stockiness $\frac{1}{2}(z_i+z_j)\cdot\log\textrm{lift}$, '
             r'restricted to balanced pairs where both elements are at '
             r'least $z \geq 2$ in genre and neither surface form is a '
             'substring of the other. Operationalizes the Indiana Jones '
             'doctrine: ordinary tropes that become identifying when '
             'stacked. Examples: "ranch + outlaw" (Western), "vampire + '
             'werewolf" (Horror), "magic + witch" (Fantasy).'),
    label='tab:combos',
    source_notebook='03c_network_and_combinations',
    column_format='llrrr', escape=False, float_format='%.2f',
)

# --- F-VI-1: Western co-occurrence network ----------------------------
cent_sub = (centrality[centrality['genre'] == ANCHOR_GENRE_FOR_NETWORK]
            .sort_values('z_ensemble_in_genre', ascending=False)
            .head(NETWORK_TOP_N).copy())
top_set = set(cent_sub['element'].tolist())

combo_sub = combinations[
    (combinations['genre']      == ANCHOR_GENRE_FOR_NETWORK) &
    (combinations['element_i'].isin(top_set)) &
    (combinations['element_j'].isin(top_set)) &
    (combinations['pmi_positive'] > 0)
].copy()
combo_sub = combo_sub.sort_values('pmi_positive', ascending=False).head(NETWORK_TOP_EDGES)

G = nx.Graph()
for _, r in cent_sub.iterrows():
    G.add_node(r['element'],
               size=float(r['z_ensemble_in_genre']),
               taxonomy=r['element_taxonomy'])
for _, r in combo_sub.iterrows():
    G.add_edge(r['element_i'], r['element_j'],
               weight=float(r['pmi_positive']))

fig, ax = plt.subplots(figsize=(8, 7.5))
pos = nx.spring_layout(G, seed=42, k=0.8, iterations=80)
node_sizes  = [max(80, 80 + 320 * G.nodes[n]['size']) for n in G.nodes]
edge_widths = [G.edges[e]['weight'] * 0.55 for e in G.edges]
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.25,
                       width=edge_widths, edge_color='#777777')
nx.draw_networkx_nodes(G, pos, ax=ax,
                       node_size=node_sizes,
                       node_color=[GENRE_COLORS[ANCHOR_GENRE_FOR_NETWORK]] * len(G.nodes),
                       edgecolors='black', linewidths=0.4)
nx.draw_networkx_labels(G, pos, labels={n: n for n in G.nodes},
                        font_size=7.5, ax=ax)
ax.set_title(f'Top {len(G.nodes)} co-occurrence network for {ANCHOR_GENRE_FOR_NETWORK}\n'
             f'Nodes sized by ensemble z-score; edges weighted by positive PMI')
ax.axis('off')
plt.tight_layout()

src = combo_sub[['element_i', 'element_j', 'pmi_positive',
                 'combination_stock', 'lift']].copy()
save_figure(
    fig, 'F-VI-1_network_western', source_data=src,
    source_notebook='03c_network_and_combinations',
    caption=(f'Co-occurrence network for the top {len(G.nodes)} stockiest '
             f'narrative elements in {ANCHOR_GENRE_FOR_NETWORK}. Nodes '
             f'sized by within-genre ensemble z-score; edges weighted by '
             f'positive PMI computed on film-level co-occurrence. '
             f'Visualizes the Indiana Jones doctrine: stock elements are '
             f'not isolated; they form recognizable clusters '
             f'(ranch/outlaw/sheriff/posse for the law-and-'
             f'order cluster in the Western; horse/saloon/town for the frontier-life '
             f'cluster) whose co-occurrence patterns themselves are '
             f'doctrinally informative. Per-genre networks for the '
             f'remaining 16 genres are in Appendix B.'),
)

# --- F-VI-2: centrality vs stockiness scatter -------------------------
cent_joined = centrality.merge(
    stockiness[['element', 'element_type', 'genre', 'ensemble']],
    on=['element', 'element_type', 'genre'], how='inner')
plot_sample = cent_joined.sample(n=min(SCATTER_SAMPLE_N, len(cent_joined)),
                                 random_state=42)

fig, ax = plt.subplots(figsize=(7.5, 5))
for g in CANONICAL_GENRES:
    sub = plot_sample[plot_sample['genre'] == g]
    ax.scatter(sub['z_ensemble_in_genre'], sub['eigenvector'],
               s=4, color=GENRE_COLORS[g], alpha=0.45,
               label=g, edgecolors='none')
ax.set_xlabel('Ensemble z-score (within genre)')
ax.set_ylabel('Eigenvector centrality')
ax.set_title('Network centrality versus element-level stockiness')
ax.set_xlim(-2, plot_sample['z_ensemble_in_genre'].quantile(0.999))
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7,
          ncol=1, frameon=False, markerscale=2)
plt.tight_layout()

corr_by_genre = (cent_joined.groupby('genre')
                 .apply(lambda d: pd.Series({
                     'corr': d['z_ensemble_in_genre'].corr(d['eigenvector']),
                     'n'   : len(d)}))
                 .reset_index())
save_figure(
    fig, 'F-VI-2_centrality_stockiness', source_data=corr_by_genre,
    source_notebook='03c_network_and_combinations',
    caption=('Eigenvector centrality versus ensemble z-score per (element, '
             'type, genre) cell. Network centrality and per-cell '
             'stockiness are correlated but not redundant: an element can '
             'sit at high stockiness without being structurally central '
             "in the genre's co-occurrence graph, and an element can be "
             'centrally positioned in the network without being among '
             'the highest-z by per-cell measures. The two views are '
             'complementary diagnostics of doctrinal stockiness.'),
)


Part VI — Combinations and networks
  tab  T-VI-1_top_combinations  (85 rows)
  fig  F-VI-1_network_western


/tmp/ipykernel_4956/20315794.py:144: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: pd.Series({


  fig  F-VI-2_centrality_stockiness


In [22]:
# ----------------------------------------------------------------------
# Part VII — Temporal dynamics
#
# Outputs:
#   T-VII-1   Vocabulary change-points per genre
#   T-VII-2   Emerging conventions by bin (CORRECTED — not all 2015)
#   F-VII-1   Per-genre bin means over time
#   F-VII-2   Canonical-element lifecycles (8 picks, NO 'haunted house')
#   F-VII-3   Per-genre survival curves
#
# Operationalizes the Restatement §18 'customary' descriptor by tracing
# the framework's measures over time. T-VII-2 corrects the
# emerging-conventions display issue from prior temporal_report.txt: the
# raw report sorts by recency and shows only 2015 emergers, but the
# actual distribution peaks at the 2000-2005 cohort. We render this as
# the true bin distribution.
# ----------------------------------------------------------------------

print('Part VII — Temporal dynamics')

# --- T-VII-1: vocabulary change-points --------------------------------
cp_sub = change_points[change_points['is_change_point']].copy()
T_VII_1 = (cp_sub[['genre', 'from_bin', 'to_bin', 'jaccard']]
           .sort_values(['genre', 'from_bin'])
           .rename(columns={'genre': 'Genre', 'from_bin': 'From bin',
                            'to_bin': 'To bin', 'jaccard': 'Jaccard'}))
save_table(
    T_VII_1, 'T-VII-1_change_points',
    caption=('Vocabulary change-points per genre: five-year bin '
             "transitions where the top-K Jaccard turnover falls in the "
             "genre's bottom decile, indicating a structural shift in the "
             "genre's conventional vocabulary. Most genres show a "
             "concentration of change-points in the early 1920s (silent-to-"
             "sound era); later change-points include Animation 1980 "
             "(Disney Renaissance), Comedy 1975, Drama 1975 + 2015, "
             "Western 2015 (post-revisionist), and Music 1940 (musical "
             "comedy era)."),
    label='tab:changepoints', source_notebook='08_temporal_analysis',
    column_format='lrrr', float_format='%.3f',
)

# --- T-VII-2: emerging conventions by bin (CORRECTED) -----------------
# The temporal_report.txt presents all emerging conventions as 2015
# emergers because it sorts by recency descending. The actual emergence
# distribution peaks at the 2000-2005 cohort. We render the true
# distribution per genre + per 5-year cohort.
emerging = element_lifecycle[
    element_lifecycle['ever_stock'] &
    (element_lifecycle['first_stock_bin'] >= 1985)
].copy()
narr_set = set(zip(classifications.loc[classifications['is_narrative_element'], 'element'],
                   classifications.loc[classifications['is_narrative_element'], 'element_type']))
emerging['is_narrative'] = emerging.apply(
    lambda r: (r['element'], r['element_type']) in narr_set, axis=1)
emerging = emerging[emerging['is_narrative']].copy()

# Pivot: rows are first_stock_bin, columns are genres, values are count
T_VII_2_long = (emerging.groupby(['first_stock_bin', 'genre']).size()
                .rename('count').reset_index())
T_VII_2 = (T_VII_2_long.pivot(index='first_stock_bin', columns='genre', values='count')
           .reindex(columns=CANONICAL_GENRES)
           .fillna(0).astype(int)
           .reset_index()
           .rename(columns={'first_stock_bin': 'First-stock bin'}))
T_VII_2['First-stock bin'] = T_VII_2['First-stock bin'].astype(int)
save_table(
    T_VII_2, 'T-VII-2_emerging_by_bin',
    caption=('Count of emerging stock conventions per genre by '
             'first-stock-crossing bin. An emerging convention is a '
             "narrative element that first crossed the z $\geq$ 1 stock "
             'threshold in a given five-year bin after 1985. Across '
             'most genres, emergence peaks in the 2000 or 2005 cohorts '
             '(post-millennium expansion of plot summaries on Wikipedia), '
             'not the 2015 cohort. This corrects a sort-order artifact in '
             "the raw temporal_report.txt that presents emerging "
             'conventions as all 2015.'),
    label='tab:emerging', source_notebook='08_temporal_analysis',
    column_format='r' + 'r' * N_GENRES, float_format='%d',
)

# --- F-VII-1: per-genre bin means over time ---------------------------
K_BIN = 50
bin_means = (stock_by_bin.sort_values(['genre', 'bin', 'ensemble'],
                                     ascending=[True, True, False])
             .groupby(['genre', 'bin']).head(K_BIN)
             .groupby(['genre', 'bin'])['ensemble'].mean().reset_index())

fig, ax = plt.subplots(figsize=(8.5, 5))
for g in CANONICAL_GENRES:
    sub = bin_means[bin_means['genre'] == g].sort_values('bin')
    ax.plot(sub['bin'], sub['ensemble'],
            color=GENRE_COLORS[g], linewidth=1.4, label=g)
ax.set_xlabel('Five-year bin (starting year)')
ax.set_ylabel(f'Mean ensemble of top-{K_BIN} per bin')
ax.set_title('Per-genre mean stockiness of top-50 elements over time')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
          fontsize=7, ncol=1, frameon=False)
plt.tight_layout()
save_figure(
    fig, 'F-VII-1_bin_means_per_genre', source_data=bin_means,
    source_notebook='08_temporal_analysis',
    caption=('Per-genre mean ensemble z-score of the top-50 stock elements '
             'in each five-year bin from 1915 to 2015. The rise in '
             'apparent stockiness over time partly reflects corpus '
             'composition (post-2000 bins contain more films per bin, '
             'producing larger within-bin spreads), not pure convention '
             'strength. The qualitative pattern (high relative '
             'stockiness in the studio era, peak around 2005-2010, '
             'partial-bin drop at 2015) is real; the absolute scale '
             'should not be interpreted as a direct measure of how '
             'conventional a genre has become.'),
)

# --- F-VII-2: canonical-element lifecycles (8 panels) -----------------
fig, axes = plt.subplots(2, 4, figsize=(11.5, 5.5))
src_rows = []
for ax, (elem, typ, gen) in zip(axes.flat, LIFECYCLE_EXAMPLES):
    sub = stock_by_bin[(stock_by_bin['element']      == elem) &
                       (stock_by_bin['element_type'] == typ) &
                       (stock_by_bin['genre']        == gen)].sort_values('bin')
    if len(sub) == 0:
        ax.text(0.5, 0.5, f'(no data)\n{elem} [{typ}] in {gen}',
                ha='center', va='center', fontsize=8, transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        continue
    ax.plot(sub['bin'], sub['ensemble'],
            color=GENRE_COLORS[gen], linewidth=1.5,
            marker='o', markersize=3)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title(f'{elem} ({typ})\nin {gen}', fontsize=9)
    ax.set_xlabel('Bin', fontsize=8)
    ax.set_ylabel('Ensemble z', fontsize=8)
    s2 = sub[['bin', 'ensemble']].copy()
    s2['element'] = elem; s2['type'] = typ; s2['genre'] = gen
    src_rows.append(s2)
plt.suptitle('Canonical element lifecycles across five-year bins', y=1.03)
plt.tight_layout()

src_df = pd.concat(src_rows, ignore_index=True) if src_rows else pd.DataFrame()
save_figure(
    fig, 'F-VII-2_canonical_lifecycles', source_data=src_df,
    source_notebook='08_temporal_analysis',
    caption=('Lifecycle trajectories of eight canonical stock elements '
             'across five-year bins. The panels span the seven most '
             'commonly-cited genre archetypes (sheriff/Western, '
             'vampire/Horror, dragon/Fantasy, soldier/War, '
             'detective/Crime, spaceship/Science Fiction, marriage/'
             'Romance) plus one multi-word chunk example (wagon train/'
             'Western). Soldier in War (top right) is the only canonical '
             'example with stock-quality presence across all 21 bins; '
             'wagon train (bottom-left chunk) is the canonical case of a '
             'transient mid-century Western convention. Per-genre '
             'principled top-6 panels for all 17 genres are in Appendix B.'),
)

# --- F-VII-3: per-genre survival curves -------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
for g in CANONICAL_GENRES:
    sub = survival[survival['genre'] == g].sort_values('lag_years')
    if len(sub) == 0:
        continue
    ax.plot(sub['lag_years'], sub['survival_probability'],
            color=GENRE_COLORS[g], linewidth=1.4,
            marker='o', markersize=3, label=g)
ax.set_xlabel('Lag (years)')
ax.set_ylabel('Top-50 survival probability')
ax.set_title('Per-genre survival of top-50 stock elements across lags')
ax.set_ylim(0, 1.0)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
          fontsize=7, ncol=1, frameon=False)
plt.tight_layout()
save_figure(
    fig, 'F-VII-3_survival_curves', source_data=survival,
    source_notebook='08_temporal_analysis',
    caption=('Per-genre survival of top-50 stock elements across lags of '
             '5, 10, 15, 20, 30, and 50 years. Survival probability is '
             'the conditional probability that an element above the '
             "z = 1 stock threshold at time $\tau$ in a given genre is "
             "still above threshold at time $\tau + \delta$. Drama "
             '(top line) has the highest survival across all lags, '
             'consistent with the reliance of that genre on human-universal '
             'narrative elements (mother, father, son, marriage). '
             'History (bottom line) has the lowest survival, consistent '
             'with the vocabulary in that genre turning over as historical '
             'subject matter evolves.'),
)
mean_survival_lag_20 = float(survival[survival['lag_years'] == 20]['survival_probability'].mean())
mean_survival_lag_50 = float(survival[survival['lag_years'] == 50]['survival_probability'].mean())
headline('mean_survival_lag_20', mean_survival_lag_20, '{:.1%}',
         'Mean survival probability of top-50 stock elements at 20-year lag')
headline('mean_survival_lag_50', mean_survival_lag_50, '{:.1%}',
         'Mean survival probability of top-50 stock elements at 50-year lag')


Part VII — Temporal dynamics
  tab  T-VII-1_change_points  (43 rows)
  tab  T-VII-2_emerging_by_bin  (7 rows)


<>:70: SyntaxWarning: invalid escape sequence '\g'
<>:179: SyntaxWarning: invalid escape sequence '\d'
<>:70: SyntaxWarning: invalid escape sequence '\g'
<>:179: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_4956/2087518902.py:70: SyntaxWarning: invalid escape sequence '\g'
  "narrative element that first crossed the z $\geq$ 1 stock "
/tmp/ipykernel_4956/2087518902.py:179: SyntaxWarning: invalid escape sequence '\d'
  "still above threshold at time $\tau + \delta$. Drama "


  fig  F-VII-1_bin_means_per_genre
  fig  F-VII-2_canonical_lifecycles
  fig  F-VII-3_survival_curves


In [23]:
# ----------------------------------------------------------------------
# Part VIII — Litigation tool
#
# Outputs:
#   T-VIII-1   Candidate scènes à faire per genre (HEADLINE LITIGATION TABLE)
#   T-VIII-2   Worked example: hypothetical Western copyright dispute
#
# T-VIII-1 is the manuscript's central product for actual litigation
# use: a per-genre table of elements that satisfy all three criteria
# (high stockiness AND narrative classification AND temporal
# durability). T-VIII-2 walks through a constructed Western case as if
# a court were applying the framework to specific disputed elements.
# ----------------------------------------------------------------------

print('Part VIII — Litigation tool')

# --- T-VIII-1: candidate scènes à faire -------------------------------
# Three inclusion criteria:
#  (1) Ensemble z above 95th within-genre percentile
#  (2) Narrative-classified by 03b (+ blocklist correction)
#  (3) Present in at least 4 five-year bins (sustained convention)
percentiles = (stockiness.groupby('genre')['ensemble']
               .quantile(LITIGATION_PCT_THRESHOLD).rename('threshold').reset_index())
candidates = stockiness.merge(percentiles, on='genre')
candidates = candidates[candidates['ensemble'] >= candidates['threshold']]

narrative_set = set(zip(
    classifications.loc[classifications['is_narrative_element'], 'element'],
    classifications.loc[classifications['is_narrative_element'], 'element_type'],
))
mask = candidates.apply(
    lambda r: (r['element'], r['element_type']) in narrative_set, axis=1)
candidates = candidates[mask].copy()

durability = (stock_by_bin.groupby(['element', 'element_type', 'genre'])
              .size().rename('n_bins').reset_index())
candidates = candidates.merge(durability,
                              on=['element', 'element_type', 'genre'], how='left')
candidates['n_bins'] = candidates['n_bins'].fillna(0).astype(int)
candidates = candidates[candidates['n_bins'] >= LITIGATION_DURABILITY_MIN_BINS].copy()

candidates = candidates.sort_values(['genre', 'ensemble'], ascending=[True, False])
candidates = candidates.drop_duplicates(['genre', 'element'])
top20 = candidates.groupby('genre').head(LITIGATION_TOP_PER).copy()
top20['rank'] = top20.groupby('genre').cumcount() + 1

T_VIII_1 = (top20.pivot_table(index='rank', columns='genre',
                              values='element', aggfunc='first')
            .reindex(columns=CANONICAL_GENRES)
            .reset_index().rename(columns={'rank': 'Rank'}))
save_table(
    T_VIII_1, 'T-VIII-1_litigation_candidates',
    caption=('Per-genre top-20 candidate scènes à faire for litigation use. '
             'Inclusion criteria: ensemble z above the 95th within-genre '
             'percentile, narrative classification by Notebook 03b '
             '(corrected by the franchise blocklist), and presence in at '
             'least four five-year bins (sustained convention, not '
             "transient artifact). This is the framework's central product "
             'for actual litigation: an element appearing on this table '
             'for its genre constitutes presumptive evidence that the '
             'element is a scènes à faire of that genre. Surface-form '
             'dedup applied within each genre.'),
    label='tab:litigation', source_notebook='09_deliverables',
    column_format='l' + 'l' * N_GENRES, escape=True,
)
n_candidates = int(len(top20))
headline('n_litigation_candidates', n_candidates, '{:,}',
         'Total candidate scenes a faire across 17 genres')

# --- T-VIII-2: worked example case ------------------------------------
# For each disputed element in the constructed case, look up the
# framework's outputs in the appropriate genre and assemble a row.

# Helper: pull the per-(element, type, genre) row from stockiness
worked_rows = []
case_genre = WORKED_CASE_GENRE
for elem, typ in WORKED_CASE_ELEMENTS:
    # Per-cell stockiness
    sub_stk = stockiness_classified[
        (stockiness_classified['element']      == elem) &
        (stockiness_classified['element_type'] == typ) &
        (stockiness_classified['genre']        == case_genre)
    ]
    # Lifecycle
    sub_life = element_lifecycle[
        (element_lifecycle['element']      == elem) &
        (element_lifecycle['element_type'] == typ) &
        (element_lifecycle['genre']        == case_genre)
    ]

    if len(sub_stk) == 0:
        worked_rows.append({
            'Element'          : f'{elem} ({typ})',
            'Ensemble z'       : float('nan'),
            'Classification'   : '(not in corpus)',
            'Durability (bins)': 0,
            'First stock'      : '--',
            'Last stock'       : '--',
            'Verdict'          : 'Insufficient data',
        })
        continue
    r = sub_stk.iloc[0]
    z = float(r['ensemble'])
    cls_label = r['element_taxonomy'] + (' (narrative)' if r['is_narrative_element'] else ' (filtered)')
    if len(sub_life):
        l = sub_life.iloc[0]
        durab = int(l['stock_duration_bins'])
        first = int(l['first_stock_bin']) if pd.notna(l['first_stock_bin']) else '--'
        last  = int(l['last_stock_bin'])  if pd.notna(l['last_stock_bin']) else '--'
    else:
        durab, first, last = 0, '--', '--'

    # Verdict logic — six tiers to differentiate the elements'
    # doctrinal posture. The finer thresholds give the manuscript a
    # concrete demonstration of the framework's discriminatory power
    # across clearly-stock, moderate, borderline, transient, filtered,
    # and insufficient cases.
    if z >= 10 and r['is_narrative_element'] and durab >= 8:
        verdict = 'Strong scènes à faire (presumptive)'
    elif z >= 5 and r['is_narrative_element'] and durab >= 4:
        verdict = 'Moderate scènes à faire (likely)'
    elif z >= 2 and r['is_narrative_element'] and durab >= 4:
        verdict = 'Borderline scènes à faire (debatable)'
    elif z >= 2 and r['is_narrative_element']:
        verdict = 'Transient — high stockiness but low durability'
    elif z >= 2:
        verdict = f'Filtered ({r["element_taxonomy"]}) — not scènes à faire'
    else:
        verdict = 'Insufficient stockiness'

    worked_rows.append({
        'Element'          : f'{elem} ({typ})',
        'Ensemble z'       : round(z, 2),
        'Classification'   : cls_label,
        'Durability (bins)': durab,
        'First stock'      : first,
        'Last stock'       : last,
        'Verdict'          : verdict,
    })

T_VIII_2 = pd.DataFrame(worked_rows)
save_table(
    T_VIII_2, 'T-VIII-2_worked_example_western',
    caption=(f'Worked example: {WORKED_CASE_NAME}. A constructed copyright '
             f'dispute in which the defendant moves for summary dismissal '
             f'on scènes à faire grounds for ten elements of the Western '
             f"plaintiff's screenplay. For each disputed element the "
             f'framework outputs (i) ensemble z-score within the Western '
             f'genre, (ii) Notebook 03b narrative classification, (iii) '
             f'durability (count of five-year bins of stock-quality '
             f'presence), (iv) the bins where the element first and last '
             f'crossed the stock threshold, and (v) a six-tier verdict on '
             f"the element's doctrinal posture. The example is "
             f"deliberately constructed to span the full verdict "
             f'spectrum: clearly-stock conventions (sheriff, outlaw, '
             f'ranch, cowboy, saloon), moderately stock (wagon train, '
             f'gunslinger), filtered place names (Texas), filtered '
             f'franchise terms (Lone Ranger), and insufficient '
             f'stockiness (chair). The framework discriminates among '
             f'these doctrinally distinct cases, which dueling-expert '
             f'testimony in the current scènes à faire practice cannot.'),
    label='tab:worked-example', source_notebook='09_deliverables',
    column_format='lrlrrr l', escape=True,
)


Part VIII — Litigation tool
  tab  T-VIII-1_litigation_candidates  (20 rows)
  tab  T-VIII-2_worked_example_western  (10 rows)


In [24]:
# ----------------------------------------------------------------------
# Part IX — Discussion: the Animation anomaly diagnostic
#
# Output:
#   T-IX-1   Animation anomaly: filtered elements closest to centroid
#
# Animation is the one genre where Arm 2 shows filtered > narrative
# (Cohen's d = -0.19). The diagnostic table shows the top filtered
# elements in Animation that sit closest to the Animation centroid in
# Voyage-4-large embedding space — confirming the doctrinal
# interpretation that Animation's semantic identity is partly carried by
# franchise characters that 03b correctly classifies as filtered.
# ----------------------------------------------------------------------

print('Part IX — Discussion: Animation anomaly diagnostic')

# Pull genre_centroids for Animation, filter to non-narrative elements,
# sort by cosine to centroid descending. Top 20 by cosine.
gc_anim_filt = genre_centroids[
    (genre_centroids['genre'] == 'Animation') &
    (~genre_centroids['is_narrative_element'])
].copy()

if 'cosine_to_centroid' in gc_anim_filt.columns:
    T_IX_1 = (gc_anim_filt.sort_values('cosine_to_centroid', ascending=False)
              .head(20)
              [['element', 'element_type', 'element_taxonomy', 'cosine_to_centroid']]
              .rename(columns={
                  'element'           : 'Filtered element',
                  'element_type'      : 'Type',
                  'element_taxonomy'  : 'Taxonomy',
                  'cosine_to_centroid': 'Cosine to Animation centroid',
              }))
else:
    # Defensive: if the column isn't there, just save a note
    T_IX_1 = pd.DataFrame([
        {'note': 'genre_centroids did not carry cosine_to_centroid column; '
                 'rebuild Notebook 05 Phase F to add it.'}])

save_table(
    T_IX_1, 'T-IX-1_animation_anomaly',
    caption=("Animation anomaly diagnostic. The 20 filtered (non-narrative) "
             "elements in the Animation genre that sit closest to the "
             "Animation centroid in Voyage-4-large embedding space. "
             "Contrary to the expectation that the Animation Arm 2 anomaly "
             "(filtered > narrative, $d = -0.19$) would be driven by "
             "franchise characters, the top-cosine filtered elements are "
             "instead meta-discourse phrases used in Animation plot "
             "summaries: 'entire plot', 'movie plot', 'plot synopsis', "
             "'hilarious scene', 'fight scene', 'previous cartoon'. These "
             "chunks are correctly tagged by Notebook 03b as "
             "'generic_or_empty' (filtered) but they appear with unusual "
             "frequency in Animation plot summaries, where the writing "
             "style is more self-referential and meta-narrative than in "
             "other genres. The Animation anomaly is therefore a property "
             "of the corpus writing style, not a defect of either the "
             "classification or the framework's element-level measures. "
             "A future paper-time refinement could extend the 03b "
             "classifier to detect and filter genre-specific discourse "
             "vocabulary like these phrases."),
    label='tab:animation-anomaly', source_notebook='05_embeddings_and_topics',
    column_format='lllr', float_format='%.3f',
)


Part IX — Discussion: Animation anomaly diagnostic
  tab  T-IX-1_animation_anomaly  (20 rows)


In [25]:
# ----------------------------------------------------------------------
# Appendices A + B — per-type tables + per-genre fan-out figures
#
# Appendix A:
#   T-App-A1  Top-25 nouns per genre
#   T-App-A2  Top-25 chunks per genre
#   T-App-A3  Top-25 verbs per genre
#   T-App-A4  Top-25 entities per genre
#
# Appendix B:
#   F-App-B01..B17  Per-genre co-occurrence networks (17 figures)
#   F-App-B18..B34  Per-genre top-6 lifecycle panels (17 figures)
# ----------------------------------------------------------------------

print('Appendices A + B')

# --- Helpers ---------------------------------------------------------
def safe_genre_slug(g):
    return str(g).replace(' ', '_').replace('/', '_')

# Appendix A: top-25 per type per genre (4 sub-tables) ----------------
for idx, (label, df_top) in enumerate([
    ('nouns'   , top_nouns),
    ('chunks'  , top_chunks),
    ('verbs'   , top_verbs),
    ('entities', top_entities),
], start=1):
    if 'rank' in df_top.columns:
        sub = df_top[df_top['rank'] <= TOP_K_PER_TYPE].copy()
    else:
        sub = (df_top.sort_values(['genre', 'score'], ascending=[True, False])
               .groupby('genre').head(TOP_K_PER_TYPE).copy())
        sub['rank'] = sub.groupby('genre').cumcount() + 1
    wide = (sub.pivot_table(index='rank', columns='genre',
                            values='element', aggfunc='first')
            .reindex(columns=CANONICAL_GENRES)
            .reset_index()
            .rename(columns={'rank': 'Rank'}))
    table_id = f'T-App-A{idx}_top25_{label}'
    save_table(
        wide, table_id,
        caption=(f'Top 25 {label} per genre by ensemble z-score, raw rankings '
                 f'without the narrative-vs-filtered cut. Provided for '
                 f'transparency; the post-filter view for body tables '
                 f'(T-IV-1, T-VIII-1) uses the narrative-only ranking.'),
        label=f'tab:top25-{label}', source_notebook='03_stockiness',
        column_format='l' + 'l' * N_GENRES, escape=True,
    )

# Appendix B1: per-genre co-occurrence networks ----------------------
def build_genre_network(genre, top_n=NETWORK_TOP_N, top_edges=NETWORK_TOP_EDGES):
    cent_sub = (centrality[centrality['genre'] == genre]
                .sort_values('z_ensemble_in_genre', ascending=False)
                .head(top_n).copy())
    top_set = set(cent_sub['element'].tolist())
    combo_sub = combinations[
        (combinations['genre']      == genre) &
        (combinations['element_i'].isin(top_set)) &
        (combinations['element_j'].isin(top_set)) &
        (combinations['pmi_positive'] > 0)
    ].copy()
    combo_sub = combo_sub.sort_values('pmi_positive', ascending=False).head(top_edges)

    G = nx.Graph()
    for _, r in cent_sub.iterrows():
        G.add_node(r['element'],
                   size=float(r['z_ensemble_in_genre']),
                   taxonomy=r['element_taxonomy'])
    for _, r in combo_sub.iterrows():
        G.add_edge(r['element_i'], r['element_j'],
                   weight=float(r['pmi_positive']))
    return cent_sub, combo_sub, G


for i, genre in enumerate(tqdm(CANONICAL_GENRES,
                               desc='F-App-B per-genre networks'), start=1):
    cent_sub, combo_sub, G = build_genre_network(genre)
    if len(G.nodes) == 0:
        print(f'  skipping {genre}: no centrality rows')
        continue
    fig, ax = plt.subplots(figsize=(8, 7.5))
    pos = nx.spring_layout(G, seed=42, k=0.8, iterations=80)
    node_sizes  = [max(80, 80 + 320 * G.nodes[n]['size']) for n in G.nodes]
    edge_widths = [G.edges[e]['weight'] * 0.55 for e in G.edges]
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.25,
                           width=edge_widths, edge_color='#777777')
    nx.draw_networkx_nodes(G, pos, ax=ax,
                           node_size=node_sizes,
                           node_color=[GENRE_COLORS[genre]] * len(G.nodes),
                           edgecolors='black', linewidths=0.4)
    nx.draw_networkx_labels(G, pos, labels={n: n for n in G.nodes},
                            font_size=7.5, ax=ax)
    ax.set_title(f'Top {len(G.nodes)} co-occurrence network for {genre}\n'
                 f'Nodes sized by ensemble z; edges weighted by positive PMI')
    ax.axis('off')
    plt.tight_layout()

    fig_id = f'F-App-B{i:02d}_network_{safe_genre_slug(genre)}'
    src = combo_sub[['element_i', 'element_j', 'pmi_positive',
                     'combination_stock', 'lift']].copy()
    save_figure(
        fig, fig_id, source_data=src,
        source_notebook='03c_network_and_combinations',
        caption=(f'Co-occurrence network for the top {len(G.nodes)} stockiest '
                 f'narrative elements in {genre}. Edge weights are positive '
                 f'PMI computed on film-level co-occurrence. Companion to '
                 f'the body figure F-VI-1 (Western network).'),
    )

# Appendix B2: per-genre top-6 lifecycle panels ----------------------
# Use the morph-aware filter: narrative + raw_stock_bins >= 3 + dedup
_raw_stock_count = (stock_by_bin[stock_by_bin['ensemble'] >= 1.0]
                    .groupby(['element', 'element_type', 'genre'])
                    .size().rename('raw_stock_bins').reset_index())
life_filt = element_lifecycle.copy()
life_filt['is_narr'] = life_filt.apply(
    lambda r: (r['element'], r['element_type']) in narrative_set, axis=1)
life_filt = life_filt.merge(_raw_stock_count,
                            on=['element', 'element_type', 'genre'], how='left')
life_filt['raw_stock_bins'] = life_filt['raw_stock_bins'].fillna(0).astype(int)
life_filt = life_filt[life_filt['is_narr'] &
                      (life_filt['raw_stock_bins'] >= 3) &
                      life_filt['peak_ensemble'].notna()].copy()

per_genre_top = (life_filt.sort_values(['genre', 'peak_ensemble'],
                                       ascending=[True, False])
                 .drop_duplicates(['genre', 'element'])
                 .groupby('genre').head(LIFECYCLE_TOP_PER_GENRE))

for i, genre in enumerate(tqdm(CANONICAL_GENRES,
                               desc='F-App-B per-genre lifecycles'),
                          start=N_GENRES + 1):
    sub_top = per_genre_top[per_genre_top['genre'] == genre]
    if len(sub_top) == 0:
        print(f'  skipping {genre}: no qualifying lifecycle rows')
        continue

    n_panels = len(sub_top)
    ncols = 3
    nrows = max(1, (n_panels + ncols - 1) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(9, 2.4 * nrows))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    src_rows = []
    for ax, (_, row) in zip(axes_flat, sub_top.iterrows()):
        elem, typ = row['element'], row['element_type']
        trace = stock_by_bin[(stock_by_bin['element']      == elem) &
                             (stock_by_bin['element_type'] == typ) &
                             (stock_by_bin['genre']        == genre)].sort_values('bin')
        if len(trace) == 0:
            ax.text(0.5, 0.5, '(no data)', ha='center', va='center',
                    fontsize=8, transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            continue
        ax.plot(trace['bin'], trace['ensemble'],
                color=GENRE_COLORS[genre], linewidth=1.5,
                marker='o', markersize=3)
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_title(f'{elem} ({typ})', fontsize=9)
        ax.set_xlabel('Bin', fontsize=8)
        ax.set_ylabel('Ensemble z', fontsize=8)
        s2 = trace[['bin', 'ensemble']].copy()
        s2['element'] = elem; s2['type'] = typ; s2['genre'] = genre
        src_rows.append(s2)
    for ax in axes_flat[len(sub_top):]:
        ax.axis('off')

    plt.suptitle(f'{genre}: top-{LIFECYCLE_TOP_PER_GENRE} stock elements '
                 f'by peak ensemble across five-year bins', y=1.02, fontsize=10)
    plt.tight_layout()

    fig_id = f'F-App-B{i:02d}_lifecycle_{safe_genre_slug(genre)}'
    src_df = pd.concat(src_rows, ignore_index=True) if src_rows else pd.DataFrame()
    save_figure(
        fig, fig_id, source_data=src_df,
        source_notebook='08_temporal_analysis',
        caption=(f'Per-genre principled lifecycles for {genre}. Top-'
                 f'{LIFECYCLE_TOP_PER_GENRE} narrative-classified elements '
                 f'by peak smoothed ensemble z, traced across five-year '
                 f'bins. Inclusion criteria: narrative classification and '
                 f'at least three bins of raw (un-smoothed) stock-quality '
                 f'presence. Companion to body figure F-VII-2.'),
    )


Appendices A + B
  tab  T-App-A1_top25_nouns  (25 rows)
  tab  T-App-A2_top25_chunks  (25 rows)
  tab  T-App-A3_top25_verbs  (25 rows)
  tab  T-App-A4_top25_entities  (25 rows)


F-App-B per-genre networks:   0%|          | 0/17 [00:00<?, ?it/s]

  fig  F-App-B01_network_Action
  fig  F-App-B02_network_Adventure
  fig  F-App-B03_network_Animation
  fig  F-App-B04_network_Comedy
  fig  F-App-B05_network_Crime
  fig  F-App-B06_network_Drama
  fig  F-App-B07_network_Family
  fig  F-App-B08_network_Fantasy
  fig  F-App-B09_network_History
  fig  F-App-B10_network_Horror
  fig  F-App-B11_network_Music
  fig  F-App-B12_network_Mystery
  fig  F-App-B13_network_Romance
  fig  F-App-B14_network_Science_Fiction
  fig  F-App-B15_network_Thriller
  fig  F-App-B16_network_War
  fig  F-App-B17_network_Western


F-App-B per-genre lifecycles:   0%|          | 0/17 [00:00<?, ?it/s]

  fig  F-App-B18_lifecycle_Action
  fig  F-App-B19_lifecycle_Adventure
  fig  F-App-B20_lifecycle_Animation
  fig  F-App-B21_lifecycle_Comedy
  fig  F-App-B22_lifecycle_Crime
  fig  F-App-B23_lifecycle_Drama
  fig  F-App-B24_lifecycle_Family
  fig  F-App-B25_lifecycle_Fantasy
  fig  F-App-B26_lifecycle_History
  fig  F-App-B27_lifecycle_Horror
  fig  F-App-B28_lifecycle_Music
  fig  F-App-B29_lifecycle_Mystery
  fig  F-App-B30_lifecycle_Romance
  fig  F-App-B31_lifecycle_Science_Fiction
  fig  F-App-B32_lifecycle_Thriller
  fig  F-App-B33_lifecycle_War
  fig  F-App-B34_lifecycle_Western


In [26]:
# ----------------------------------------------------------------------
# Export — manifest, headline JSON + LaTeX macros, caption seeds,
#           paper-section index, build summary
# ----------------------------------------------------------------------

print('Export: writing manifest, headlines, caption seeds, section index, summary')

# --- Manifest --------------------------------------------------------
manifest_df = pd.DataFrame(MANIFEST_ROWS)
manifest_df.to_csv(OUT_DIR / 'manifest.csv', index=False)
print(f'  manifest.csv             {len(manifest_df)} entries')

# --- Headline JSON ---------------------------------------------------
with open(OUT_DIR / 'headline_numbers.json', 'w', encoding='utf-8') as f:
    json.dump(HEADLINE, f, indent=2, default=str)
print(f'  headline_numbers.json    {len(HEADLINE)} entries')

# --- Headline TeX macros ---------------------------------------------
tex_lines = [
    '% Auto-generated by Notebook 09. Do not edit by hand.',
    '% Source: v4_rebuild/09_deliverables/headline_numbers.json',
    '%',
    r'% Usage in manuscript: \PartialRCorpuswide expands to the current value.',
    '',
]
for key, info in HEADLINE.items():
    val = info['value']
    fmt = info['fmt']
    desc = info.get('description', '')
    try:
        val_str = fmt.format(val) if fmt else str(val)
    except Exception:
        val_str = str(val)
    val_tex = val_str.replace('%', r'\%').replace('#', r'\#')
    macro = to_macro_name(key)
    tex_lines.append(f'\\newcommand{{\\{macro}}}{{{val_tex}}}% {desc}')
with open(OUT_DIR / 'headline_numbers.tex', 'w', encoding='utf-8') as f:
    f.write('\n'.join(tex_lines) + '\n')
print(f'  headline_numbers.tex     {len(HEADLINE)} macros')

# --- Caption seeds ---------------------------------------------------
cap_lines = [
    '% Caption seeds for Notebook 09 outputs.',
    '% Each entry: a stable identifier and the caption text to paste',
    '% into the corresponding figure or table environment in Overleaf.',
    '%',
    '% Usage:',
    '%   \\begin{figure}',
    '%     \\includegraphics[width=\\textwidth]{F-IV-1_measure_correlation.pdf}',
    '%     \\caption{<paste caption here from this file>}',
    '%     \\label{fig:F-IV-1}',
    '%   \\end{figure}',
    '',
]
for fig_id, caption in sorted(CAPTION_SEEDS.items()):
    cap_lines.append(f'%% {fig_id}')
    cap_lines.append(caption)
    cap_lines.append('')
with open(OUT_DIR / 'caption_seeds.tex', 'w', encoding='utf-8') as f:
    f.write('\n'.join(cap_lines) + '\n')
print(f'  caption_seeds.tex        {len(CAPTION_SEEDS)} captions')

# --- Paper section index ---------------------------------------------
n_figs = sum(1 for r in MANIFEST_ROWS if r['kind'] == 'figure')
n_tabs = sum(1 for r in MANIFEST_ROWS if r['kind'] == 'table')

# Map each output identifier to its manuscript section.
# Examples:
#   T-IV-1_top10_per_genre   -> 'Part IV'
#   F-VII-1_bin_means        -> 'Part VII'
#   T-App-A1_top25_nouns     -> 'Appendix A'
#   F-App-B01_network_Action -> 'Appendix B'
def manuscript_section(ident):
    # Strip the descriptor suffix: 'T-IV-1_top10_per_genre' -> 'T-IV-1'
    prefix = ident.split('_')[0]
    parts = prefix.split('-')
    # parts = ['T', 'IV', '1'] or ['F', 'App', 'B01']
    if len(parts) >= 3 and parts[1] == 'App':
        letter = parts[2][0]  # 'A' from 'A1', 'B' from 'B01'
        return f'Appendix {letter}'
    elif len(parts) >= 2:
        return f'Part {parts[1]}'
    return 'Unknown'


# Sort key that puts Parts in roman-numeral order, then appendices
_ROMAN_ORDER = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6,
                'VII': 7, 'VIII': 8, 'IX': 9, 'X': 10}
def section_sort_key(section_name):
    if section_name.startswith('Part '):
        roman = section_name.replace('Part ', '')
        return (0, _ROMAN_ORDER.get(roman, 99), section_name)
    elif section_name.startswith('Appendix '):
        return (1, section_name[-1], section_name)
    return (2, 0, section_name)


# Output-sort key within a section: tables before figures, then by id
def output_sort_key(row):
    return (0 if row['kind'] == 'table' else 1, row['id'])


# Group manifest entries by manuscript section
section_to_rows = {}
for row in MANIFEST_ROWS:
    sec = manuscript_section(row['id'])
    section_to_rows.setdefault(sec, []).append(row)

section_index = [
    '# Notebook 09 — Paper Section Index',
    '',
    'A reference mapping each output to its manuscript location.',
    'Outputs are grouped by manuscript Part (II through Appendix B). Within',
    'each section, tables are listed before figures.',
    '',
]

for sec in sorted(section_to_rows.keys(), key=section_sort_key):
    rows_in_sec = sorted(section_to_rows[sec], key=output_sort_key)
    section_index.append(f'## {sec}')
    section_index.append(f'_{len(rows_in_sec)} outputs_')
    section_index.append('')
    for row in rows_in_sec:
        kind = row['kind']
        ext = '.pdf' if kind == 'figure' else '.tex'
        cap = row['caption']
        if len(cap) > 200:
            cap = cap[:200].rsplit(' ', 1)[0] + '...'
        section_index.append(f"- `{row['id']}{ext}` ({kind}) — {cap}")
    section_index.append('')

with open(OUT_DIR / 'paper_section_index.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(section_index) + '\n')
print(f'  paper_section_index.md   {len(section_to_rows)} sections')

# --- Build summary ---------------------------------------------------
lines = [
    '======================================================================',
    'NOTEBOOK 09 -- DELIVERABLES BUILD SUMMARY (manuscript-oriented rebuild)',
    '======================================================================',
    f'figures rendered  : {n_figs}',
    f'tables rendered   : {n_tabs}',
    f'headline macros   : {len(HEADLINE)}',
    f'caption seeds     : {len(CAPTION_SEEDS)}',
    f'section index entries: {len(section_to_rows)}',
    '',
    'Per-section output count:',
]
section_counts = {}
for r in MANIFEST_ROWS:
    sec = manuscript_section(r['id'])
    section_counts[sec] = section_counts.get(sec, 0) + 1
for sec in sorted(section_counts.keys(), key=section_sort_key):
    lines.append(f'  {sec:20s} {section_counts[sec]} outputs')

lines += [
    '',
    '======================================================================',
    'HEADLINE NUMBERS REGISTERED',
    '======================================================================',
]
for key, info in HEADLINE.items():
    val = info['value']
    fmt = info['fmt'] or '{}'
    try:
        val_str = fmt.format(val)
    except Exception:
        val_str = str(val)
    desc = info.get('description', '')
    lines.append(f'  {key:40s} = {val_str:15s} -- {desc}')

lines += [
    '',
    '======================================================================',
    'OUTPUT TREE',
    '======================================================================',
]
total_bytes = 0
for sub in sorted(OUT_DIR.rglob('*')):
    if sub.is_file():
        rel = sub.relative_to(OUT_DIR)
        size = sub.stat().st_size
        total_bytes += size
        lines.append(f'  {str(rel):60s}  ({size:>10,} bytes)')
lines.append('')
lines.append(f'  TOTAL: {total_bytes:,} bytes')

with open(OUT_DIR / 'build_summary.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines) + '\n')

print('\nBuild complete.')
print(f'  figures: {n_figs}')
print(f'  tables : {n_tabs}')
print(f'  macros : {len(HEADLINE)}')
print(f'  output : {OUT_DIR}')


Export: writing manifest, headlines, caption seeds, section index, summary
  manifest.csv             65 entries
  headline_numbers.json    23 entries
  headline_numbers.tex     23 macros
  caption_seeds.tex        65 captions
  paper_section_index.md   10 sections

Build complete.
  figures: 46
  tables : 19
  macros : 23
  output : /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/09_deliverables


## Outputs produced

Under `v4_rebuild/09_deliverables/`:

- **`figures/pdf/`** — vector PDFs for `\includegraphics`. Filenames carry
  the section identifier (`F-IV-1_measure_correlation.pdf` belongs in
  Part IV, position 1). Body figures plus 34 appendix per-genre fan-out
  figures (17 networks + 17 lifecycles).
- **`figures/png/`** — 300-DPI raster previews.
- **`figures/source_data/`** — CSV behind each plot. A coauthor can
  re-skin a figure without rerunning upstream notebooks.
- **`tables/latex/`** — `\input`-ready booktabs tables. Each declares
  its own `\caption` and `\label`. Wide tables (T-IV-1, T-VIII-1, the
  App-A series) need `\begin{sidewaystable}` or `\resizebox` wrapping.
- **`tables/csv/`** — CSV mirrors for archival use.
- **`headline_numbers.json`** — every named number registered above.
- **`headline_numbers.tex`** — `\newcommand` macros built from the JSON.
  `\input{headline_numbers.tex}` in the Overleaf preamble; manuscript
  writes `\PartialRCorpuswide` or `\CohensDMeanArmOne` and the values
  stay in sync.
- **`caption_seeds.tex`** — ready-to-paste `\caption{...}` strings per
  artifact. Avoids rewriting captions for every figure.
- **`paper_section_index.md`** — a manuscript-writer reference mapping
  each section identifier to its outputs.
- **`manifest.csv`** — provenance audit trail (kind, id, source notebook,
  caption).
- **`build_summary.txt`** — inventory, per-section counts, headline-number
  listing, full output tree with sizes.

## How to wire this into the Overleaf project

1. Copy `figures/pdf/` and `tables/latex/` into the Overleaf project.
2. Copy `headline_numbers.tex` and `caption_seeds.tex` into the Overleaf
   project (recommend a `data/` subfolder for both).
3. Add to the preamble:
   ```latex
   \input{data/headline_numbers.tex}
   ```
4. In the manuscript, reference figures and tables by their identifier:
   ```latex
   \begin{table}[htbp]
       \centering
       \input{tables/T-IV-1_top10_per_genre.tex}
   \end{table}
   ```
5. For figures, use the section identifier filename:
   ```latex
   \begin{figure}[htbp]
       \centering
       \includegraphics[width=\textwidth]{figures/F-V-1_length_confound.pdf}
       \caption{...}  % copy the caption text from caption_seeds.tex
       \label{fig:V-1}
   \end{figure}
   ```
6. Reference numbers in prose via macros:
   ```
   Mean Cohen's d across the seventeen genres was \CohensDMeanArmOne,
   with \NGenresArmOneLarge of seventeen genres reaching the large-effect
   threshold d $\geq$ 0.8.
   ```
7. Wide tables (T-IV-1, T-VIII-1, T-App-A1..A4) overflow portrait page
   width. Wrap each in `\begin{sidewaystable}` or
   `\resizebox{\textwidth}{!}{...}` for the manuscript build.

## Known caveats for the manuscript writer

- **F-VII-1 absolute scale.** The per-genre bin-mean ensemble figure
  shows rising stockiness over time, but this partly reflects corpus
  composition (post-2000 bins have more films per bin, producing larger
  within-bin spreads). The qualitative pattern is real, the absolute
  scale should not be over-interpreted.
- **Animation Arm 2 anomaly (Part IX).** The framework is correct;
  Animation's corpus is franchise-dense enough that filtered items sit
  close to the genre centroid. T-IX-1 provides the diagnostic table.
- **F-VI-1 anchor genre.** Only the Western network is in the body;
  remaining 16 are in Appendix B. The choice of Western for the anchor
  reflects the genre's iconic role in scènes à faire doctrine (the
  doctrine's leading cases involve Western and similar film genres)
  and its tight, clean network structure.
- **Multi-POS retention.** A surface form like 'german' may appear in
  multiple element_type rows (noun + entity_NORP) for the same
  conceptual element. This is intentional per the pipeline design
  (CLAUDE.md, "Bug 4"); body tables show the highest-ensemble row per
  surface form to avoid display redundancy.
- **Franchise blocklist.** The blocklist contains 28 entries documented
  in the setup cell with one-line rationales. Each entry corrects a
  03b classification error confirmed in the post-canonicalization data.
  Reviewers can audit the list directly.

## What feeds where

| Manuscript section | Tables | Figures | Macros |
|---|---|---|---|
| Methods (Part III) | T-III-1, T-III-2, T-III-3 | F-III-1 | NFilmsTotal, NUniqueElementsPreFilter, NCellsFiltered, NElementsFiltered, NNarrativeElements, ChunkPrecision, PooledRecall, NValidatedFilms |
| Core results (Part IV) | T-IV-1, T-IV-2 | F-IV-1, F-IV-2, F-IV-3 | MeanMeasureCorrelation, NarrativeShare |
| Validation (Part V) | T-V-1, T-V-2, T-V-3 | F-V-1, F-V-2, F-V-3 | PartialRCorpuswide, RawRCorpuswide, NGenresNegativePartialR, CohensDMeanArmOne, CohensDMedianArmOne, NGenresArmOneSig, NGenresArmOneLarge, CohensDMeanArmTwo, NGenresArmTwoNarrGreater |
| Combinations (Part VI) | T-VI-1 | F-VI-1, F-VI-2 | — |
| Temporal (Part VII) | T-VII-1, T-VII-2 | F-VII-1, F-VII-2, F-VII-3 | MeanSurvivalLagTwoZero, MeanSurvivalLagFiveZero |
| Litigation tool (Part VIII) | T-VIII-1, T-VIII-2 | — | NLitigationCandidates |
| Discussion (Part IX) | T-IX-1 | — | — |
| Appendix A | T-App-A1..A4 | — | — |
| Appendix B | — | F-App-B01..B34 | — |
